# CS-781 Phase 4 (Court Judgment Prediction) — Architecture 4: RetNet (138M, 12 layers, d768), old BPE with the original regex - Phase-1 BERTScore 0.8327

**Task.** One binary outcome (0/1) per case, scored with Macro-F1. Submission = `prediction.json` (a list of
`{"id": ..., "pred": 0|1}`) at the root of a zip.

**Model (hierarchical).** Every case gets the **exact Phase-1 preprocessing + tokenizer**. I keep the **first 512-token
chunk** (head: parties, court, what is appealed) and the **last 7 chunks** (tail: arguments and reasoning; ILDC removed the
final decision sentences), so ≈4k tokens per case. Shorter cases are kept whole. Every chunk goes through my Phase-1 RetNet
(bidirectional retention, + an adaptation method) and is mean-pooled into a chunk vector, plus a chunk-position embedding.
A **BiGRU over the chunks**, then attention and max pooling, then a linear layer give P(1). Training samples 4 chunks per case
(always the last one) for speed; prediction uses all of them. The 0/1 threshold is tuned on the clean dev set.

**Data hygiene (important).**
* The released `dev.json` contains **all 1517 test cases with labels**, plus every real dev case twice.
* `train.json` contains ILDC-single inside ILDC-multi.
* I drop every dev or train row whose id or text is in test, de-duplicate by text, and **never read the test labels**.
* What remains: ≈31.7k train cases and **994 clean dev cases (497/497)**. Dev is used only for model selection, the threshold and analysis. There is no train+dev refit.

**Experiments** (one Codabench zip each; Phase-4 technique list):

| job | technique | note |
|---|---|---|
| `lorma` | LoRMA (multiplicative low-rank) | **compulsory**; KD teacher; also runs the section ablation |
| `kd` | knowledge distillation: LoRMA teacher → 6-layer student | **compulsory** |
| `adalora` | AdaLoRA (`peft`) | adaptive-rank low-rank |
| `bitfit` | BitFit (biases + norms) | sparse PEFT |
| `adapter` | Houlsby adapters | bottleneck adapters |
| `partial` | top-4 blocks fine-tuned | partial / layer-wise FT |
| `gradual` | gradual unfreezing (head → top 2 → 4 → 6 blocks) | layer freezing/unfreezing strategy |
| `frozen` | frozen Phase-1 encoder, only the document head trained | probe / ablation |


In [1]:
# ============================================================ 1. SETUP
import os, sys, json, time, shutil, subprocess, threading, pickle
from pathlib import Path
T_START = time.time()

QUICK_TEST = False       # True: tiny end-to-end check (~20-30 min).  False: the real run (commit it).
GROUP = "B"            # "ALL": all jobs on this account | "A" / "B": half of the jobs each (run on two accounts)
SESSION_H = 12.0         # Kaggle session limit; jobs are shortened automatically so the notebook always finishes
ARCH, NORM = "ret", "clean"
PHASE1_HINT = "archit-4-renet"   # substring of the attached Phase-1 notebook's input path

WORK = Path("/kaggle/working/p4"); WORK.mkdir(parents=True, exist_ok=True)
OUT  = WORK / ("runs_quick" if QUICK_TEST else "runs")

def find(name, must_contain="", anywhere=False):
    hits = sorted(p for p in Path("/kaggle/input").rglob(name) if (anywhere or PHASE1_HINT in str(p).lower()) and must_contain in str(p))
    return str(hits[0]) if hits else None

PHASE1_CKPT = find("best.pt", "ckpt") or find("best.pt") or find("last.pt")      # retnet/ckpt/best.pt
MODEL_PY    = find("ret_model.py")
TOK_PY      = find("a1_tokenizer.py") or find("a1_tokenizer.py", anywhere=True)
TOK_PATH    = find("bpe_tokenizer.json") or find("bpe_tokenizer.json", anywhere=True)
print("checkpoint :", PHASE1_CKPT); print("model code :", MODEL_PY); print("tokenizer  :", TOK_PY, "|", TOK_PATH)
assert PHASE1_CKPT and MODEL_PY, f"attach the Phase-1 notebook whose input path contains '{PHASE1_HINT}' (Add Input -> Your Work -> Notebooks), or fix PHASE1_HINT"
assert TOK_PY and TOK_PATH, "tokenizer files not found - attach the tokenizer dataset (bpe_tokenizer.json + a1_tokenizer.py)"

# Phase-4 data: a local copy if one is attached, otherwise download it from Hugging Face with my HF secret
# (to the HF cache outside /kaggle/working, so the private data never ends up in this notebook's output).
hits = sorted(Path("/kaggle/input").rglob("Phase_4_CJPE/Train/train.json"))
if hits:
    DATA = hits[0].parent.parent; print("using local copy:", DATA)
else:
    HF_TOKEN = None
    try:
        from kaggle_secrets import UserSecretsClient
        for k in ["HF", "HF_TOKEN"]:
            try: HF_TOKEN = UserSecretsClient().get_secret(k); break
            except Exception: pass
    except ImportError:
        HF_TOKEN = os.environ.get("HF_TOKEN")
    assert HF_TOKEN, "add your Hugging Face token as a Kaggle secret named HF (Add-ons -> Secrets) and tick it for this notebook"
    from huggingface_hub import snapshot_download
    DATA = Path(snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=HF_TOKEN,
                                  allow_patterns=["Phase_4_CJPE/*"])) / "Phase_4_CJPE"
print("data:", DATA, sorted(str(p.relative_to(DATA)) for p in DATA.rglob("*.json")))
try:
    import peft; print("peft", peft.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=False)


checkpoint : /kaggle/input/notebooks/alex595786/archit-4-renet/retnet/ckpt/best.pt
model code : /kaggle/input/notebooks/alex595786/archit-4-renet/retnet/ret_model.py
tokenizer  : /kaggle/input/notebooks/alex595786/archit-4-renet/retnet/a1_tokenizer.py | /kaggle/input/datasets/nirajkumarprasad/p4-a4-prev-dataset/bpe_tokenizer.json


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

data: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_4_CJPE ['Test/test.json', 'Train/dev.json', 'Train/train.json']
peft 0.20.0


In [2]:
%%writefile /kaggle/working/p4/p2_ner.py
"""CS-781 Phase 2 (L-NER) on top of my Phase-1 models: Transformer-XL (arch=txl), decoder-only LegalLM (arch=dec),
the encoder of the encoder-decoder (arch=encdec), or RetNet (arch=ret).

One file, used two ways:
  * imported by the notebook  -> data prep, alignment checks, model-equivalence test
  * run as a script            -> `python p2_ner.py job.json` trains/evaluates/predicts ONE experiment

Pipeline
  text --(ASCII-fold + whitespace-collapse, exactly like Phase-1 pretraining, but keeping a char map)-->
  BPE tokens with ORIGINAL character offsets --> BIO tags (25) --> 512-token sliding windows -->
  Transformer-XL encoder (+ PEFT) + token head --> overlap-weighted log-prob merge per document -->
  BIO-constrained Viterbi --> character spans (whitespace-stripped) --> prediction.json

Adaptation methods (METHOD):
  full     full fine-tuning
  partial  top-N blocks + head (layer-wise / partial fine-tuning)
  lora     LoRA:  y = W x + s*B(A x)
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025, multiplicative):  y = (I + s*B A) W x   (B = 0 -> identity at init)
  adapter  Houlsby-style bottleneck adapters after attention and FFN sub-layers
  bitfit   biases (+ norms; TXL u/v biases). Bias-free LegalLM: zero-init biases added to every projection
  kd       knowledge distillation: a fine-tuned 14-layer teacher -> a shallower student built from Phase-1 layers
"""
import json
import math
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np

# ============================================================== text normalisation with char map
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")
# Phase-1 pre-tokenizer WITH the underscore fix that the Phase-1 notebook applied
PRE_RE = re.compile(r" ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+")


def normalize_with_map(text):
    """ASCII mode (Transformer-XL runs): same output as the TXL Phase-1 `normalize_text` (ASCII fold,
    straight quotes, single spaces, no newlines), plus omap[i] = ORIGINAL index of normalised char i."""
    norm, st, _ = normalize_ascii_map(text)
    return norm, st


def normalize_ascii_map(text):
    out, ost, oen = [], [], []
    pend = -1                                   # orig index of the first whitespace in a pending run
    for i, ch in enumerate(text):
        s = ch
        if not ch.isascii():
            s = ch.translate(_MAP)
            if not s.isascii():
                s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
        if s and _CTRL.search(s):
            s = _CTRL.sub("", s)
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


def normalize_nfkc_map(text, delete_table):
    """NFKC mode (decoder-only run): same output as a1_tokenization.normalize_text
    (delete Cc/Cf/Cs -> NFKC -> collapse whitespace -> strip). NFKC is applied per *cluster*
    (a char + its combining marks) so compositions match the whole-string NFKC.
    ost/oen[i] = ORIGINAL [start, end) of the cluster that produced normalised char i."""
    out, ost, oen = [], [], []
    pend = -1
    n, i = len(text), 0
    while i < n:
        j = i + 1
        while j < n and unicodedata.combining(text[j]):
            j += 1
        s = unicodedata.normalize("NFKC", text[i:j].translate(delete_table))
        for c in s:
            if c.isspace():
                if pend < 0:
                    pend = i
            else:
                if pend >= 0 and out:
                    out.append(" "); ost.append(pend); oen.append(pend + 1)
                pend = -1
                out.append(c); ost.append(i); oen.append(j)
        i = j
    return "".join(out), ost, oen


def align_map(orig, norm):
    """Character map norm -> orig from a diff (used only when the fast cluster map disagrees)."""
    import difflib
    st, en = [0] * len(norm), [0] * len(norm)
    sm = difflib.SequenceMatcher(None, orig, norm, autojunk=False)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        for j in range(j1, j2):
            if tag == "equal":
                st[j], en[j] = i1 + (j - j1), i1 + (j - j1) + 1
            else:                                  # replaced / inserted: map to the whole changed region
                st[j], en[j] = i1, max(i2, i1 + 1) if i2 > i1 else min(i1 + 1, len(orig))
    return st, en


# RetNet Phase-1 preprocessing: ret_data.clean_text = remove "\r", collapse 3+ newlines to 2, strip (unicode kept)
_NL3 = re.compile(r"\n{3,}")
# the ORIGINAL a1_tokenizer pre-tokenizer (no underscore fix: "_" is never matched, i.e. dropped) - used by the RetNet run
PRE_RE_ORIG = re.compile(r" ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+")


def clean_text_ret(t):
    return _NL3.sub("\n\n", t.replace("\r", "")).strip()


def normalize_clean_map(text):
    idx = [i for i, c in enumerate(text) if c != "\r"]
    keep, run = [], 0
    for i in idx:                                              # 3+ consecutive newlines -> keep the first two
        if text[i] == "\n":
            run += 1
            if run > 2:
                continue
        else:
            run = 0
        keep.append(i)
    a, b = 0, len(keep)
    while a < b and text[keep[a]].isspace():
        a += 1
    while b > a and text[keep[b - 1]].isspace():
        b -= 1
    keep = keep[a:b]
    return "".join(text[i] for i in keep), keep, [i + 1 for i in keep]


# Archit-2-v2 preprocessing: " ".join(text.split())  (whitespace runs -> one space, ends stripped, unicode kept)
def normalize_ws_map(text):
    out, ost, oen = [], [], []
    pend = -1
    for i, ch in enumerate(text):
        if ch.isspace():
            if pend < 0:
                pend = i
        else:
            if pend >= 0 and out:
                out.append(" "); ost.append(pend); oen.append(pend + 1)
            pend = -1
            out.append(ch); ost.append(i); oen.append(i + 1)
    return "".join(out), ost, oen


class HFBPE:
    """Archit-2-v2 tokenizer: HuggingFace `tokenizers` ByteLevelBPETokenizer(vocab.json, merges.txt)
    (<pad>0 <s>1 </s>2 <unk>3), text = " ".join(text.split()). Offsets come from the tokenizer itself."""

    def __init__(self, tok_path, norm="ws"):
        from tokenizers import ByteLevelBPETokenizer
        p = Path(tok_path)
        p = p.parent if p.is_file() else p
        self.tok = ByteLevelBPETokenizer(str(p / "vocab.json"), str(p / "merges.txt"))
        self.kind, self.norm = "hf", norm
        self.vocab_size = self.tok.get_vocab_size()
        assert [self.tok.token_to_id(t) for t in ("<pad>", "<s>", "</s>", "<unk>")] == [0, 1, 2, 3]

    def normalize(self, text):
        n, st, en = normalize_ws_map(text)
        assert n == " ".join(text.split())
        return n, st, en

    def reference_encode(self, raw_text):
        return self.tok.encode(" ".join(raw_text.split())).ids


# ============================================================== tokenizer with offsets
class BPE:
    """Uniform wrapper around my two from-scratch Phase-1 tokenizers.
      kind='old' : a1_tokenizer.py  + bpe_tokenizer.json   (<pad>0 <unk>1 <s>2 </s>3; underscore-fixed regex)
      kind='new' : a1_tokenization.py + vocab.json/merges.txt (<pad>0 <s>1 </s>2 <unk>3; GPT-2 style regex)
    norm='ascii' : TXL pretraining preprocessing (ASCII fold)   |   norm='nfkc' : a1_tokenization.normalize_text
    norm='clean' : RetNet preprocessing (\r removal, newline collapse; ORIGINAL old regex, underscores dropped)
    Exposes: pre_re, encode_piece(piece) -> ids, nbytes[id], vocab_size, normalize(text), reference_encode(text)."""

    def __init__(self, tokenizer_py, tok_path, norm="ascii"):
        import importlib.util
        name = Path(tokenizer_py).stem
        spec = importlib.util.spec_from_file_location(name + "_p2", tokenizer_py)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        self.mod, self.norm = mod, norm
        if hasattr(mod, "BPETokenizer"):                              # old tokenizer
            self.kind = "old"
            p = Path(tok_path)
            self.tok = mod.BPETokenizer.load(str(p if p.is_file() else p / "bpe_tokenizer.json"))
            self.pre_re = PRE_RE_ORIG if norm == "clean" else PRE_RE      # RetNet run used the unpatched regex
            self.encode_piece = self.tok._encode_pretoken
            self.nbytes = [len(b) for b in self.tok.id_to_bytes]
            self.vocab_size = self.tok.get_vocab_size()
        else:                                                         # new tokenizer
            self.kind = "new"
            p = Path(tok_path)
            self.tok = mod.Tokenizer.load(str(p.parent if p.is_file() else p))
            self.pre_re = mod.PRETOKEN_RE
            self.encode_piece = self.tok._encode_word
            k = mod.N_SPECIAL
            self.nbytes = [0] * k + [len(t) for t in self.tok.id_to_token[k:]]   # 1 char = 1 byte (GPT-2 byte map)
            self.vocab_size = self.tok.vocab_size
        assert (norm == "ascii") or (norm == "nfkc" and self.kind == "new") or (norm == "clean" and self.kind == "old")

    def normalize(self, text):
        if self.norm == "nfkc":
            n, st, en = normalize_nfkc_map(text, self.mod._DELETE_TABLE)
            ref = self.mod.normalize_text(text)
            if n != ref:                       # rare: NFKC interaction across clusters -> exact diff alignment
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                st, en = align_map(text, ref)
                n = ref
            return n, st, en
        if self.norm == "clean":
            n, st, en = normalize_clean_map(text)
            if n != clean_text_ret(text):
                self.n_fallback = getattr(self, "n_fallback", 0) + 1
                n = clean_text_ret(text)
                st, en = align_map(text, n)
            return n, st, en
        return normalize_ascii_map(text)

    def encode_full(self, norm_text):
        out = []
        for piece in self.pre_re.findall(norm_text):
            out.extend(self.encode_piece(piece))
        return out

    def reference_encode(self, raw_text):
        """Exactly what Phase-1 pretraining fed the model for this raw text (used only for checks)."""
        if self.norm == "nfkc":
            return self.tok.encode(raw_text)                         # normalize_text + encode
        if self.norm == "clean":                                     # ret_data.clean_text + original a1_tokenizer.encode
            t = clean_text_ret(raw_text)
            return [i for piece in PRE_RE_ORIG.findall(t) for i in self.tok._encode_pretoken(piece)]
        return self.encode_full(self.normalize(raw_text)[0])


def load_bpe(tokenizer_py, tok_path, norm="ascii"):
    if norm == "ws":
        return HFBPE(tok_path, norm)
    return BPE(tokenizer_py, tok_path, norm)


def encode_doc(tok, text):
    """-> dict(ids int32[n], cs int32[n], ce int32[n]).  For token t, the ORIGINAL-text span is [cs, ce),
    where cs skips the single leading space a BPE token may carry. Handles tokens that split a
    multi-byte UTF-8 character (byte-level BPE) by mapping byte offsets to characters."""
    norm, ost, oen = tok.normalize(text)
    ids, cs, ce = [], [], []
    if getattr(tok, "kind", "") == "hf":                         # offsets straight from the HF tokenizer
        if not norm:
            return dict(ids=np.zeros(0, np.int32), cs=np.zeros(0, np.int32), ce=np.zeros(0, np.int32))
        enc = tok.tok.encode(norm)
        for tid, (a, z) in zip(enc.ids, enc.offsets):
            z = max(z, a + 1) - 1                                # last normalised char touched by this token
            a = min(a, len(norm) - 1); z = min(max(z, a), len(norm) - 1)
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid); cs.append(ost[a2]); ce.append(oen[z])
        return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))
    for m in tok.pre_re.finditer(norm):
        piece, p0 = m.group(), m.start()
        if piece.isascii():
            b2c = None
            nb = len(piece)
        else:
            b2c = [ci for ci, ch in enumerate(piece) for _ in range(len(ch.encode("utf-8")))]
            nb = len(b2c)
        bp = 0
        for tid in tok.encode_piece(piece):
            L = tok.nbytes[tid]
            a = p0 + (bp if b2c is None else b2c[bp])
            z = p0 + ((bp + L - 1) if b2c is None else b2c[bp + L - 1])     # last char touched by this token
            bp += L
            a2 = a + 1 if (norm[a] == " " and z > a) else a
            ids.append(tid)
            cs.append(ost[a2])
            ce.append(oen[z])
        assert bp == nb, "token byte lengths do not add up"
    return dict(ids=np.asarray(ids, np.int32), cs=np.asarray(cs, np.int32), ce=np.asarray(ce, np.int32))


# ============================================================== labels / BIO
def make_tags(labels):
    tags = ["O"] + [f"B-{l}" for l in labels] + [f"I-{l}" for l in labels]
    return tags, {t: i for i, t in enumerate(tags)}


def strip_span(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def bio_from_gold(text, enc, ents, tag2id):
    """Token BIO ids from character-level gold entities (entities are whitespace-stripped first)."""
    owner = np.full(len(text) + 1, -1, np.int32)
    lab_of = []
    for k, e in enumerate(ents):
        s, en = strip_span(text, int(e[0]), int(e[1]))
        owner[s:en] = k
        lab_of.append(e[2])
    cs = enc["cs"]
    y = np.zeros(len(cs), np.int64)
    prev = -1
    for t in range(len(cs)):
        k = owner[cs[t]] if cs[t] < len(text) else -1
        if k >= 0 and text[cs[t]].isspace() and k != prev:     # a whitespace-only token continues an entity, never starts one
            k = -1
        if k >= 0:
            y[t] = tag2id[("I-" if k == prev else "B-") + lab_of[k]]
        prev = k
    return y


def spans_from_tags(text, enc, tag_ids, tags):
    out, cur = [], None                     # cur = [first_tok, last_tok, label]
    for t, ti in enumerate(tag_ids):
        tg = tags[ti]
        if tg == "O":
            if cur: out.append(cur); cur = None
            continue
        bi, lab = tg.split("-", 1)
        if bi == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [t, t, lab]
        else:
            cur[1] = t
    if cur: out.append(cur)
    res = []
    for a, b, lab in out:
        s, e = strip_span(text, int(enc["cs"][a]), int(enc["ce"][b]))
        if e > s:
            res.append([s, e, lab])
    return res


def bio_transitions(tags):
    """allowed[i, j] for tag i -> tag j; start_ok[j]."""
    T = len(tags)
    allowed = np.ones((T, T), bool)
    start_ok = np.ones(T, bool)
    for j, tj in enumerate(tags):
        if tj.startswith("I-"):
            lab = tj[2:]
            start_ok[j] = False
            for i, ti in enumerate(tags):
                allowed[i, j] = ti in (f"B-{lab}", f"I-{lab}")
    return allowed, start_ok


def viterbi(logp, allowed, start_ok):
    """logp [n, T] log-probs -> best BIO-valid tag sequence."""
    n, T = logp.shape
    if n == 0:
        return np.zeros(0, np.int64)
    trans = np.where(allowed, 0.0, -1e9)
    score = np.where(start_ok, logp[0], -1e9)
    back = np.zeros((n, T), np.int32)
    for t in range(1, n):
        cand = score[:, None] + trans            # [from, to]
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    best = np.zeros(n, np.int64)
    best[-1] = int(score.argmax())
    for t in range(n - 1, 0, -1):
        best[t - 1] = back[t, best[t]]
    return best


# ============================================================== metric (strict span + label)
def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = {(int(s), int(e), l) for s, e, l, *_ in gold[did]}
        p = {(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = dict(p=P, r=R, f1=F, tp=tp[l], fp=fp[l], fn=fn[l])
    mac = lambda k: sum(per[l][k] for l in labels) / len(labels)
    return dict(macro_f1=mac("f1"), macro_p=mac("p"), macro_r=mac("r"), per_label=per)


def error_breakdown(gold, pred):
    """Classify every false positive: label error (same span, other label), boundary error
    (overlaps a gold span of the same label), or spurious; plus the label-confusion counts."""
    kinds, conf = Counter(), Counter()
    for did in gold:
        g = [(int(s), int(e), l) for s, e, l, *_ in gold[did]]
        gs = set(g)
        for s, e, l in [(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])]:
            if (s, e, l) in gs:
                continue
            same = [gl for (a, b, gl) in g if a == s and b == e]
            if same:
                kinds["label_error"] += 1; conf[f"{same[0]}->{l}"] += 1
            elif any(a < e and s < b and gl == l for a, b, gl in g):
                kinds["boundary_error"] += 1
            elif any(a < e and s < b for a, b, gl in g):
                kinds["overlap_other_label"] += 1
            else:
                kinds["spurious"] += 1
    return dict(fp_kinds=dict(kinds), top_confusions=conf.most_common(15))


# ============================================================== data prep (called from the notebook)
def prepare(data_dir, tok, labels, cache_path):
    data_dir = Path(data_dir)
    tags, tag2id = make_tags(labels)
    out = {}
    for fold in ["fold1", "fold2", "fold3"]:
        sub = "Test" if fold == "fold3" else "Train"
        texts = json.load(open(data_dir / sub / f"{fold}-text.json", encoding="utf-8"))
        gold = json.load(open(data_dir / sub / f"{fold}-gold.json", encoding="utf-8")) if fold != "fold3" else None
        docs = {}
        for did, text in texts.items():
            enc = encode_doc(tok, text)
            d = dict(text=text, **enc)
            if gold is not None:
                d["gold"] = gold.get(did, [])
                d["y"] = bio_from_gold(text, enc, d["gold"], tag2id)
            docs[did] = d
        out[fold] = docs
    pickle.dump(dict(labels=labels, tags=tags, folds=out), open(cache_path, "wb"))
    return out


def alignment_report(folds, labels):
    """Upper bound of strict macro-F1 if the model predicted the gold BIO tags perfectly."""
    tags, _ = make_tags(labels)
    rep = {}
    for fold in ["fold1", "fold2"]:
        gold = {d: v["gold"] for d, v in folds[fold].items()}
        pred = {d: spans_from_tags(v["text"], v, v["y"], tags) for d, v in folds[fold].items()}
        sc = strict_scores(gold, pred, labels)
        miss = []
        for d, v in folds[fold].items():
            p = {tuple(x) for x in pred[d]}
            for s, e, l, *_ in v["gold"]:
                if (s, e, l) not in p:
                    miss.append((d, s, e, l, v["text"][max(0, s - 15):e + 15]))
        rep[fold] = dict(upper_bound_macro_f1=sc["macro_f1"], n_gold=sum(len(v["gold"]) for v in folds[fold].values()),
                         n_unrecoverable=len(miss), examples=miss[:12])
    ntok = {f: int(sum(len(v["ids"]) for v in folds[f].values())) for f in folds}
    rep["tokens_per_fold"] = ntok
    return rep


# ============================================================== torch parts
import torch                                   # noqa: E402
import torch.nn as nn                          # noqa: E402
import torch.nn.functional as F                # noqa: E402


def _import(path, name):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def load_phase1(txl_model_py, ckpt_path):
    """Transformer-XL Phase-1 checkpoint: {'model': state_dict, 'config': dict}."""
    mod = _import(txl_model_py, "txl_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    cfg = mod.TXLConfig(**ck["config"])
    m = mod.TransformerXL(cfg)
    m.load_state_dict(ck["model"])
    return m, mod


def load_decoder(legal_lm_py, weights_path):
    """Decoder-only LegalLM Phase-1 weights (run1/final/model.pt + model_config.json)."""
    mod = _import(legal_lm_py, "legal_lm_p2")
    m = mod.load_legal_lm(weights_path, device="cpu", dtype=torch.float32)
    return m.train(), mod


def load_encdec(edmodel_py, ckpt_path):
    """Encoder-decoder Phase-1 checkpoint (phase1_encdec_v2/best.pt: {'model', 'cfg', ...})."""
    mod = _import(edmodel_py, "edmodel_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.EncDecLM(mod.EDConfig(**ck["cfg"]))
    m.load_state_dict(ck["model"])
    return m, mod


def load_retnet(ret_model_py, ckpt_path):
    """RetNet Phase-1 checkpoint (retnet/ckpt/best.pt: {'model', 'config': {'model': ...}, 'vocab_size', ...})."""
    mod = _import(ret_model_py, "ret_model_p2")
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = mod.RetNet(ck["vocab_size"], **ck["config"]["model"])
    sd = ck["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    m.load_state_dict(sd)
    return m, mod


# ---- Archit-2-v2 encoder-decoder (same module/parameter names as the Phase-1 notebook, generation code removed)
class _A2Emb(nn.Module):
    def __init__(self, vocab_size, d_model, max_length, pad_id, dropout):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, ids, offset=0):
        T = ids.size(1)
        pos = torch.arange(offset, offset + T, device=ids.device).unsqueeze(0)
        return self.dropout(self.token_embedding(ids) * math.sqrt(self.d_model) + self.position_embedding(pos))


class _A2MHA(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        self.d_model, self.h, self.dh, self.p = d_model, num_heads, d_model // num_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.dh).transpose(1, 2)

    def forward(self, query, key_value=None, key_padding_mask=None, causal=False):
        kv = query if key_value is None else key_value
        q, k, v = self._split(self.q_proj(query)), self._split(self.k_proj(kv)), self._split(self.v_proj(kv))
        mask = None
        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].bool()
            if causal:
                cm = torch.tril(torch.ones(query.size(1), kv.size(1), dtype=torch.bool, device=query.device))[None, None]
                mask = mask & cm
                causal = False
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=self.p if self.training else 0.0, is_causal=causal)
        B, _, T, _ = o.shape
        return self.out_proj(o.transpose(1, 2).contiguous().view(B, T, self.d_model))


class _A2FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(self.dropout(self.activation(self.fc1(x))))


class _A2EncLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.ffn, self.dropout2 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, src_mask):
        x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=src_mask))
        return x + self.dropout2(self.ffn(self.norm2(x)))


class _A2DecLayer(nn.Module):
    def __init__(self, d, h, d_ff, p):
        super().__init__()
        self.norm1, self.self_attn, self.dropout1 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm2, self.cross_attn, self.dropout2 = nn.LayerNorm(d), _A2MHA(d, h, p), nn.Dropout(p)
        self.norm3, self.ffn, self.dropout3 = nn.LayerNorm(d), _A2FFN(d, d_ff, p), nn.Dropout(p)

    def forward(self, x, enc_out, src_mask, self_mask=None):
        """self_mask=None -> causal (= Phase-1). Otherwise a key-padding mask -> bidirectional decoder."""
        if self_mask is None:
            x = x + self.dropout1(self.self_attn(self.norm1(x), causal=True))
        else:
            x = x + self.dropout1(self.self_attn(self.norm1(x), key_padding_mask=self_mask))
        x = x + self.dropout2(self.cross_attn(self.norm2(x), key_value=enc_out, key_padding_mask=src_mask))
        return x + self.dropout3(self.ffn(self.norm3(x)))


class _A2Stack(nn.Module):
    def __init__(self, emb, layers, d):
        super().__init__()
        self.embedding, self.layers, self.final_norm = emb, nn.ModuleList(layers), nn.LayerNorm(d)


class A2V2EncDec(nn.Module):
    def __init__(self, vocab_size=32000, d_model=768, heads=12, enc_layers=6, dec_layers=8, d_ff=3072,
                 dropout=0.05, enc_max=1024, dec_max=256, pad=0):
        super().__init__()
        self.cfg = dict(vocab_size=vocab_size, d_model=d_model, heads=heads, enc_layers=enc_layers,
                        dec_layers=dec_layers, d_ff=d_ff, enc_max=enc_max, dec_max=dec_max)
        ee = _A2Emb(vocab_size, d_model, enc_max, pad, dropout)
        de = _A2Emb(vocab_size, d_model, dec_max, pad, dropout)
        de.token_embedding.weight = ee.token_embedding.weight
        self.encoder = _A2Stack(ee, [_A2EncLayer(d_model, heads, d_ff, dropout) for _ in range(enc_layers)], d_model)
        self.decoder = _A2Stack(de, [_A2DecLayer(d_model, heads, d_ff, dropout) for _ in range(dec_layers)], d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = de.token_embedding.weight

    def encode(self, ids, src_mask):
        x = self.encoder.embedding(ids)
        for l in self.encoder.layers:
            x = l(x, src_mask)
        return self.encoder.final_norm(x)

    def decode_hidden(self, ids, enc, src_mask):
        x = self.decoder.embedding(ids)
        for l in self.decoder.layers:
            x = l(x, enc, src_mask)
        return self.decoder.final_norm(x)


def load_encdec2(ckpt_path):
    """Archit-2-v2 best_model.pt (a plain state_dict). Shapes are read from the weights (v2: 768/6/8/3072; v1: 512/6/6/2048)."""
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model" in sd and not any(k.startswith("encoder.") for k in sd):
        sd = sd["model"]
    if any(k.startswith("module.") for k in sd):
        sd = {k[7:]: v for k, v in sd.items()}
    V, d = sd["encoder.embedding.token_embedding.weight"].shape
    ne = len({k.split(".")[2] for k in sd if k.startswith("encoder.layers.")})
    nd = len({k.split(".")[2] for k in sd if k.startswith("decoder.layers.")})
    m = A2V2EncDec(vocab_size=V, d_model=d, heads=d // 64, enc_layers=ne, dec_layers=nd,
                   d_ff=sd["encoder.layers.0.ffn.fc1.weight"].shape[0],
                   enc_max=sd["encoder.embedding.position_embedding.weight"].shape[0],
                   dec_max=sd["decoder.embedding.position_embedding.weight"].shape[0])
    m.load_state_dict(sd, strict=True)
    return m


def load_backbone(J):
    if J.get("arch", "txl") == "encdec2":
        return load_encdec2(J["phase1_ckpt"])
    if J.get("arch", "txl") == "ret":
        return load_retnet(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "encdec":
        return load_encdec(J["model_py"], J["phase1_ckpt"])[0]
    if J.get("arch", "txl") == "dec":
        return load_decoder(J["model_py"], J["phase1_ckpt"])[0]
    return load_phase1(J.get("model_py", J.get("txl_model_py")), J["phase1_ckpt"])[0]


class LowRankLinear(nn.Module):
    """mode='lora'  : y = W x + b + s * B(A x)
       mode='lorma' : y = (I + s * B A) (W x) + b       (LoRMA_+, multiplicative, identity at init)"""

    def __init__(self, base: nn.Linear, r, alpha, dropout, mode):
        super().__init__()
        self.base, self.mode, self.s = base, mode, alpha / r
        for p in base.parameters():
            p.requires_grad_(False)
        din = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, din))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == "lora":
            return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)
        y0 = F.linear(x, self.base.weight)
        y = y0 + self.s * F.linear(F.linear(self.drop(y0), self.A), self.B)
        return y if self.base.bias is None else y + self.base.bias


class Adapter(nn.Module):
    """Houlsby-style bottleneck adapter applied to the output of a sub-layer projection."""

    def __init__(self, base: nn.Linear, bottleneck):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, 0, 0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))


class BiasLinear(nn.Module):
    """BitFit for bias-free projections: frozen W plus a trainable, zero-initialised bias."""

    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad_(False)
        self.bias = nn.Parameter(torch.zeros(base.out_features))

    def forward(self, x):
        return self.base(x) + self.bias


class TaggerHead(nn.Module):
    def _init_head(self, d, n_tags, dropout, head):
        self.drop = nn.Dropout(dropout)
        self.head_type = head
        if head == "bilstm":
            self.lstm = nn.LSTM(d, 256, batch_first=True, bidirectional=True)
            self.head = nn.Linear(512, n_tags)
        else:
            self.head = nn.Linear(d, n_tags)

    def forward(self, ids, attn_mask):
        h = self.hidden(ids, attn_mask)
        if self.head_type == "bilstm":
            h, _ = self.lstm(self.drop(h))
        return self.head(self.drop(h))


class TXLTagger(TaggerHead):
    """Transformer-XL (Phase-1 weights) used as a token encoder.
    attn='bidir' : every token sees left AND right context; relative distances are extended to negative
                   values (the same sinusoid / r_net / v-bias parametrisation, just evaluated at d<0).
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.pos_emb, self.ln_f = base.emb, base.pos_emb, base.ln_f
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.final_norm = self.ln_f
        self._init_head(self.d, n_tags, dropout, head)

    def _block(self, blk, h, r, gidx, mask):
        a = blk.attn
        B, L, _ = h.shape
        H, dh = a.H, a.dh
        x = blk.ln1(h)
        q = a.q_net(x).view(B, L, H, dh).transpose(1, 2)
        kv = a.kv_net(x).view(B, L, 2, H, dh)
        k, v = kv[:, :, 0].transpose(1, 2), kv[:, :, 1].transpose(1, 2)
        rk = a.r_net(r).view(-1, H, dh).transpose(0, 1).unsqueeze(0)          # [1,H,2L-1,dh]
        qu = (q + a.u[None, :, None, :]) * a.scale
        qv = (q + a.v[None, :, None, :]) * a.scale
        AC = torch.matmul(qu, k.transpose(-1, -2))                              # [B,H,L,L]
        BDf = torch.matmul(qv, rk.transpose(-1, -2))                            # [B,H,L,2L-1]
        BD = BDf.gather(-1, gidx.expand(B, H, L, L))
        score = (AC + BD).float().masked_fill(mask, -1e30)
        prob = a.dropatt(F.softmax(score, dim=-1)).to(v.dtype)
        out = torch.matmul(prob, v).transpose(1, 2).reshape(B, L, H * dh)
        h = h + a.drop(a.o_net(out))
        h = h + blk.drop(blk.ff2(blk.drop(F.gelu(blk.ff1(blk.ln2(h))))))
        return h

    def hidden(self, ids, attn_mask):
        """ids [B,L] long, attn_mask [B,L] bool (True = real token) -> final hidden [B,L,D]."""
        B, L = ids.shape
        dev = ids.device
        h = self.emb(ids)
        pos = torch.arange(L - 1, -L, -1, device=dev, dtype=torch.float32)       # distance d = L-1-c
        r = self.pos_emb(pos)
        i = torch.arange(L, device=dev)
        gidx = ((L - 1) - i[:, None] + i[None, :])[None, None]                   # c for distance i-j
        mask = (~attn_mask)[:, None, None, :]                                    # [B,1,1,L]
        if self.attn_mode == "causal":
            mask = mask | torch.ones(L, L, dtype=torch.bool, device=dev).triu(1)[None, None]
        for blk in self.blocks:
            h = self._block(blk, h, r, gidx, mask)
        return self.ln_f(h)


class DecTagger(TaggerHead):
    """Decoder-only LegalLM (RoPE, RMSNorm, SwiGLU; Phase-1 weights) used as a token encoder.
    attn='bidir' : causal mask removed (RoPE is relative, so left and right distances are both defined)
    attn='causal': exactly the Phase-1 computation (verified equal in the notebook test)."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.cfg = base.cfg
        self.tok_emb, self.norm = base.tok_emb, base.norm
        keep = list(range(len(base.blocks))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.blocks[i] for i in keep])
        self.d = base.cfg.d_model
        self.rope_mod = sys.modules[type(base).__module__]
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.norm
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        B, T = ids.shape
        dev = ids.device
        hd = self.cfg.d_model // self.cfg.n_head
        pos = torch.arange(T, device=dev)[None]
        cos, sin = self.rope_mod.rope_cos_sin(pos, hd, self.cfg.rope_theta)
        allow = attn_mask[:, None, None, :]                                   # [B,1,1,T] True = may attend
        if self.attn_mode == "causal":
            allow = allow & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        allow = allow | torch.eye(T, dtype=torch.bool, device=dev)[None, None]  # pad rows never fully masked
        x = self.tok_emb(ids)
        for blk in self.blocks:
            a = blk.attn
            hx = blk.attn_norm(x)
            q = a.q_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            k = a.k_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            v = a.v_proj(hx).view(B, T, a.h, a.hd).transpose(1, 2)
            q, k = self.rope_mod.apply_rope(q, cos, sin), self.rope_mod.apply_rope(k, cos, sin)
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=allow)
            x = x + blk.drop(a.o_proj(y.transpose(1, 2).reshape(B, T, -1)))
            x = x + blk.drop(blk.mlp(blk.mlp_norm(x)))
        return self.norm(x)


class EncTagger(TaggerHead):
    """Phase-1 encoder-decoder as a tagger.
    use_decoder=False : encoder only (natively bidirectional).
    use_decoder=True  : BART-style. The window goes through the encoder AND, as decoder input, through the decoder
                        (self-attention + cross-attention to the encoder); tags are read from the DECODER states.
                        attn='bidir' removes the decoder's causal mask, attn='causal' keeps it (= Phase-1 computation).
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        self.cfg = base.c
        self.tok, self.enc_pos, self.enc_norm = base.tok, base.enc_pos, base.enc_norm
        ke = list(range(len(base.enc_layers))) if layers is None else [i for i in layers if i < len(base.enc_layers)]
        self.enc_blocks = nn.ModuleList([base.enc_layers[i] for i in ke])
        if use_decoder:
            self.dec_pos, self.dec_norm = base.dec_pos, base.dec_norm
            kd = list(range(len(base.dec_layers))) if layers is None else [i for i in layers if i < len(base.dec_layers)]
            self.dec_blocks = nn.ModuleList([base.dec_layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))   # for PEFT / partial
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = base.c.d_model
        for m in self.blocks.modules():
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self.emb_drop = nn.Dropout(dropout)
        self._init_head(self.d, n_tags, dropout, head)

    def encode(self, ids, attn_mask):
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.enc_pos(torch.arange(T, device=dev))[None])
        mask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal" and not self.use_decoder:
            mask = mask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        mask = mask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.enc_blocks:
            x = l(x, mask)
        return self.enc_norm(x)

    def hidden(self, ids, attn_mask):
        mem = self.encode(ids, attn_mask)
        if not self.use_decoder:
            return mem
        T = ids.size(1)
        dev = ids.device
        x = self.emb_drop(self.tok(ids) + self.dec_pos(torch.arange(T, device=dev))[None])
        cmask = attn_mask[:, None, None, :]                                     # cross-attn: real encoder tokens
        smask = attn_mask[:, None, None, :]
        if self.attn_mode == "causal":
            smask = smask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        smask = smask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.dec_blocks:
            x = x + l.drop(l.self_attn(l.n1(x), mask=smask))
            x = x + l.drop(l.cross(l.n2(x), mem=mem, mask=cmask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.dec_norm(x)


class ED2Tagger(TaggerHead):
    """Archit-2-v2 encoder-decoder as a tagger.
    use_decoder=False : tags from the ENCODER (natively bidirectional, 1024 positions). attn must be 'bidir'.
    use_decoder=True  : BART-style. Window -> encoder, same window -> decoder (cross-attends to the encoder);
                        tags from the decoder states. attn='causal' = Phase-1 computation. Decoder has 256 positions.
    KD student layers (if given) are applied to both stacks."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear", use_decoder=False):
        super().__init__()
        self.attn_mode, self.use_decoder = attn, use_decoder
        enc = base.encoder
        self.enc_emb, self.enc_norm = enc.embedding, enc.final_norm
        ke = list(range(len(enc.layers))) if layers is None else [i for i in layers if i < len(enc.layers)]
        self.enc_blocks = nn.ModuleList([enc.layers[i] for i in ke])
        if use_decoder:
            dec = base.decoder
            self.dec_emb, self.dec_norm = dec.embedding, dec.final_norm
            kd = list(range(len(dec.layers))) if layers is None else [i for i in layers if i < len(dec.layers)]
            self.dec_blocks = nn.ModuleList([dec.layers[i] for i in kd])
            self.blocks = nn.ModuleList(list(self.enc_blocks) + list(self.dec_blocks))
            self.final_norm = self.dec_norm
        else:
            self.blocks = self.enc_blocks
            self.final_norm = self.enc_norm
        self.layer_ids = layers
        self.d = enc.embedding.d_model
        for m in list(self.blocks.modules()) + [self.enc_emb.dropout] + ([self.dec_emb.dropout] if use_decoder else []):
            if isinstance(m, nn.Dropout):
                m.p = dropout
        self._init_head(self.d, n_tags, dropout, head)

    def hidden(self, ids, attn_mask):
        x = self.enc_emb(ids)
        for l in self.enc_blocks:
            x = l(x, attn_mask)
        mem = self.enc_norm(x)
        if not self.use_decoder:
            return mem
        y = self.dec_emb(ids)
        sm = None if self.attn_mode == "causal" else attn_mask
        for l in self.dec_blocks:
            y = l(y, mem, attn_mask, self_mask=sm)
        return self.dec_norm(y)


class RetTagger(TaggerHead):
    """RetNet (Phase-1 weights) as a token encoder, parallel form of multi-scale retention over a window.
    attn='causal': D_nm = gamma^(n-m) for n>=m  (exactly the Phase-1 computation; verified in the notebook)
    attn='bidir' : D_nm = gamma^|n-m|           (symmetric decay, so every token also retains its right context)
    Padding uses document id -1, so the same-document mask of RetNet also removes pads."""

    def __init__(self, base, n_tags, attn="bidir", layers=None, dropout=0.1, head="linear"):
        super().__init__()
        self.attn_mode = attn
        self.emb, self.final_ln = base.emb, base.final_ln
        self.register_buffer("inv_freq", base.inv_freq.clone(), persistent=False)
        keep = list(range(len(base.layers))) if layers is None else list(layers)
        self.layer_ids = keep
        self.blocks = nn.ModuleList([base.layers[i] for i in keep])
        self.d = base.d_model
        self.rot = sys.modules[type(base).__module__].rotate
        self.emb_drop = nn.Dropout(dropout)
        for blk in self.blocks:
            blk.drop.p = dropout
        self.final_norm = self.final_ln
        self._init_head(self.d, n_tags, dropout, head)

    def _msr(self, m, x, cos, sin, doc):
        B, L, _ = x.shape
        H, dk, dv = m.H, m.dk, m.dv
        q = m.wq(x).view(B, L, H, dk)
        k = m.wk(x).view(B, L, H, dk)
        v = m.wv(x).view(B, L, H, dv)
        g = m.wg(x)
        with torch.autocast(device_type=x.device.type, enabled=False):
            q = (self.rot(q.float(), cos, sin) * dk ** -0.5).transpose(1, 2)
            k = self.rot(k.float(), cos, sin).transpose(1, 2)
            v = v.float().transpose(1, 2)
            i = torch.arange(L, device=x.device, dtype=torch.float32)
            diff = i.view(L, 1) - i.view(1, L)
            lg = m.log_gamma.view(H, 1, 1)
            if self.attn_mode == "causal":
                D = torch.exp(lg * diff.clamp(min=0)) * (diff >= 0)
            else:
                D = torch.exp(lg * diff.abs())
            same = (doc.unsqueeze(2) == doc.unsqueeze(1)).unsqueeze(1)
            o = torch.matmul(torch.matmul(q, k.transpose(-1, -2)) * D.unsqueeze(0) * same, v)
            o = o.transpose(1, 2).reshape(B * L, H * dv)
            o = F.group_norm(o, H, m.gn_w.float(), m.gn_b.float(), 1e-5)
        o = o.view(B, L, H * dv)
        return m.wo(F.silu(g) * o.to(g.dtype))

    def hidden(self, ids, attn_mask):
        B, L = ids.shape
        doc = torch.where(attn_mask, torch.zeros_like(ids), torch.full_like(ids, -1))
        pos = torch.arange(L, device=ids.device)[None].expand(B, L)
        ang = pos.float().unsqueeze(-1) * self.inv_freq
        cos, sin = ang.cos().unsqueeze(2), ang.sin().unsqueeze(2)
        h = self.emb_drop(self.emb(ids))
        for blk in self.blocks:
            h = h + blk.drop(self._msr(blk.msr, blk.ln1(h), cos, sin, doc))
            y = blk.ln2(h)
            h = h + blk.drop(blk.down(F.silu(blk.gate(y)) * blk.up(y)))
        return self.final_ln(h)


def _linear_slots(blk, names):
    """(holder_module, attribute) for every nn.Linear attribute in `names` inside one block."""
    out = []
    for mod in blk.modules():
        for nm in names:
            if isinstance(getattr(mod, nm, None), nn.Linear) and not isinstance(mod, (LowRankLinear, Adapter, BiasLinear)):
                out.append((mod, nm))
    return out


def apply_method(model, method, J):
    """Freeze / wrap according to the adaptation method. The head is always trainable."""
    for p in model.parameters():
        p.requires_grad_(method in ("full", "kd"))
    if method == "partial":
        for blk in list(model.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in model.final_norm.parameters():
            p.requires_grad_(True)
    elif method in ("lora", "lorma"):
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                setattr(holder, nm, LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method))
    elif method == "adapter":
        for blk in model.blocks:
            for holder, nm in _linear_slots(blk, J.get("adapter_targets", ["o_net", "ff2"])):
                setattr(holder, nm, Adapter(getattr(holder, nm), J["adapter_dim"]))
    elif method == "bitfit":
        if J.get("arch", "txl") in ("dec", "ret"):            # bias-free model: add zero-init biases + norms
            for blk in model.blocks:
                for holder, nm in _linear_slots(blk, J["lowrank_targets"]):
                    setattr(holder, nm, BiasLinear(getattr(holder, nm)))
        for n, p in model.named_parameters():
            if n.endswith("bias") or n.endswith(".u") or n.endswith(".v") or n.endswith(".gn_w") or n.endswith(".gn_b"):
                p.requires_grad_(True)
        for m in model.modules():                              # every LayerNorm / RMSNorm
            if "Norm" in type(m).__name__:
                for p in m.parameters():
                    p.requires_grad_(True)
    for p in model.head.parameters():
        p.requires_grad_(True)
    if getattr(model, "lstm", None) is not None:
        for p in model.lstm.parameters():
            p.requires_grad_(True)
    return model


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# ============================================================== windows / batching
def windows_for(n, L, stride):
    if n <= L:
        return [0]
    st = list(range(0, n - L + 1, stride))
    if st[-1] + L < n:
        st.append(n - L)
    return st


def make_train_windows(docs, L, stride):
    W = []
    for d in docs.values():
        n = len(d["ids"])
        for s in windows_for(n, L, stride):
            W.append((d["ids"][s:s + L], d["y"][s:s + L]))
    return W


def collate(items, with_y=True):
    L = max(len(x[0]) for x in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    am = torch.zeros(len(items), L, dtype=torch.bool)
    y = torch.full((len(items), L), -100, dtype=torch.long)
    for b, it in enumerate(items):
        n = len(it[0])
        ids[b, :n] = torch.as_tensor(it[0], dtype=torch.long)
        am[b, :n] = True
        if with_y:
            y[b, :n] = torch.as_tensor(it[1], dtype=torch.long)
    return ids, am, y


@torch.no_grad()
def doc_logprobs(model, docs, L, stride, bs, device, amp=True):
    """Merge per-window log-probs into one [n, T] array per doc (triangular weights: tokens near a window
    edge get little weight, so every token is mostly judged with context on both sides)."""
    model.eval()
    jobs = []
    for did, d in docs.items():
        n = len(d["ids"])
        if n:
            for s in windows_for(n, L, stride):
                jobs.append((did, s, min(L, n - s)))
    T = model.head.out_features
    acc = {did: np.zeros((len(d["ids"]), T), np.float32) for did, d in docs.items()}
    wsum = {did: np.zeros((len(d["ids"]), 1), np.float32) for did, d in docs.items()}
    jobs.sort(key=lambda j: -j[2])
    for k in range(0, len(jobs), bs):
        chunk = jobs[k:k + bs]
        items = [(docs[did]["ids"][s:s + n], None) for did, s, n in chunk]
        ids, am, _ = collate(items, with_y=False)
        ids, am = ids.to(device), am.to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device.type == "cuda"):
            lg = model(ids, am)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        for b, (did, s, n) in enumerate(chunk):
            pos = np.arange(n)
            w = np.minimum(pos + 1, n - pos).astype(np.float32)[:, None]
            if s == 0:
                w[: n // 2] = np.maximum(w[: n // 2], n / 2)        # doc start: no left context exists anyway
            if s + n == len(docs[did]["ids"]):
                w[n // 2:] = np.maximum(w[n // 2:], n / 2)          # doc end
            acc[did][s:s + n] += lp[b, :n] * w
            wsum[did][s:s + n] += w
    return {did: acc[did] / np.maximum(wsum[did], 1e-6) for did in docs}


def decode_docs(docs, lps, tags, use_viterbi=True):
    allowed, start_ok = bio_transitions(tags)
    pred = {}
    for did, d in docs.items():
        lp = lps[did]
        tg = viterbi(lp, allowed, start_ok) if use_viterbi else lp.argmax(-1)
        pred[did] = spans_from_tags(d["text"], d, tg, tags)
    return pred


# ============================================================== training
def class_weights(windows, T, mode):
    if mode == "none":
        return None
    cnt = np.bincount(np.concatenate([w[1] for w in windows]), minlength=T).astype(np.float64)
    w = np.ones(T)
    ent = cnt[1:]
    mean = ent[ent > 0].mean()
    w[1:] = np.clip(np.sqrt(mean / np.maximum(ent, 1)), 0.5, 4.0)
    return torch.tensor(w, dtype=torch.float32)


def build_model(J, base_factory, n_tags, method):
    base = base_factory()
    layers = J["kd_student_layers"] if method == "kd" else None
    cls = {"dec": DecTagger, "encdec": EncTagger, "encdec2": ED2Tagger, "ret": RetTagger}.get(J.get("arch", "txl"), TXLTagger)
    kw = dict(use_decoder=J.get("encdec_use_decoder", False)) if cls in (EncTagger, ED2Tagger) else {}
    if cls is ED2Tagger and not kw["use_decoder"]:
        assert J["attn"] == "bidir", "encoder-only tagging is always bidirectional"
    m = cls(base, n_tags, attn=J["attn"], layers=layers, dropout=J["dropout"], head=J["head"], **kw)
    return apply_method(m, method, J)


def train_one(J, model, train_docs, val_docs, tags, labels, device, epochs, log, teacher=None):
    torch.manual_seed(J["seed"]); np.random.seed(J["seed"])
    model.to(device)
    W = make_train_windows(train_docs, J["max_len"], J["train_stride"])
    cw = class_weights(W, len(tags), J["class_weight"])
    cw = cw.to(device) if cw is not None else None
    head_names = {n for n, _ in model.named_parameters() if n.startswith("head") or n.startswith("lstm")}
    new_ids = set()
    for mm in model.modules():
        if isinstance(mm, LowRankLinear):
            new_ids |= {id(mm.A), id(mm.B)}
        elif isinstance(mm, Adapter):
            new_ids |= {id(p) for p in list(mm.down.parameters()) + list(mm.up.parameters())}
    new_names = {n for n, p in model.named_parameters() if id(p) in new_ids}
    g_head = [p for n, p in model.named_parameters() if p.requires_grad and (n in head_names or n in new_names)]
    g_body = [p for n, p in model.named_parameters() if p.requires_grad and n not in head_names and n not in new_names]
    groups = [dict(params=g_head, lr=J["head_lr"])]
    if g_body:
        groups.append(dict(params=g_body, lr=J["lr"], is_body=True))
    opt = torch.optim.AdamW(groups, weight_decay=J["weight_decay"])
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    steps_per_epoch = math.ceil(len(W) / J["batch_size"])
    total = steps_per_epoch * epochs
    warm = max(1, int(0.06 * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)   # low start: fewer skipped fp16 steps
    n_skip = 0
    best = dict(f1=-1, epoch=0, state=None, report=None, pred=None)
    hist, step, bad = [], 0, 0
    t0 = time.time()
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    for ep in range(1, epochs + 1):
        model.train()
        perm = np.random.permutation(len(W))
        tl, tn, gsum = 0.0, 0, [0.0]
        for k in range(0, len(W), J["batch_size"]):
            ids, am, y = collate([W[i] for i in perm[k:k + J["batch_size"]]])
            ids, am, y = ids.to(device), am.to(device), y.to(device)
            f = (step + 1) / warm if step < warm else max(0.0, (total - step) / max(1, total - warm))   # warmup + linear decay
            # LP-FT: optionally keep the pretrained body frozen for the first epoch(s) so the random head
            # does not distort the Phase-1 features, then re-warm the body learning rate.
            nf = J.get("freeze_body_epochs", 0)
            if ep <= nf:
                fb = 0.0
            else:
                bs = step - nf * steps_per_epoch
                fb = min(1.0, (bs + 1) / warm) * (1.0 if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * (fb if (g.get("is_body") and nf) else f)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                lg = model(ids, am)
            lg = lg.float()
            loss = F.cross_entropy(lg.view(-1, lg.size(-1)), y.view(-1), weight=cw, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    tl_ = teacher(ids, am).float()
                Tt = J["kd_temperature"]
                kl = F.kl_div(F.log_softmax(lg / Tt, -1), F.log_softmax(tl_ / Tt, -1), log_target=True, reduction="none").sum(-1)
                kl = (kl * am).sum() / am.sum()
                loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * (Tt ** 2) * kl
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            # clip each parameter group separately: a large body gradient must never shrink the head's update
            gn = [float(torch.nn.utils.clip_grad_norm_(g["params"], 1.0)) for g in opt.param_groups]
            gsum = [a + b for a, b in zip(gsum, gn)] if tn else gn
            sc0 = scaler.get_scale() if use_amp else 1.0
            scaler.step(opt); scaler.update()
            if use_amp and scaler.get_scale() < sc0:
                n_skip += 1                                     # overflow -> this optimizer step was skipped
            tl += loss.item(); tn += 1; step += 1
        rec = dict(epoch=ep, train_loss=tl / max(tn, 1), secs=time.time() - t0, skipped_steps=n_skip,
                   grad_norm_head=round(gsum[0] / max(tn, 1), 3),
                   grad_norm_body=round(gsum[1] / max(tn, 1), 3) if len(gsum) > 1 else None)
        if val_docs:
            lps = doc_logprobs(model, val_docs, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
            pred = decode_docs(val_docs, lps, tags)
            sc = strict_scores({d: v["gold"] for d, v in val_docs.items()}, pred, labels)
            rec.update(val_f1=sc["macro_f1"], val_p=sc["macro_p"], val_r=sc["macro_r"])
            if sc["macro_f1"] > best["f1"]:
                best.update(f1=sc["macro_f1"], epoch=ep, report=sc, pred=pred,
                            state={n: p.detach().cpu().clone() for n, p in model.state_dict().items()})
                bad = 0
            else:
                bad += 1
        hist.append(rec)
        log(json.dumps(rec))
        if val_docs and bad >= J["patience"]:
            log(f"early stop at epoch {ep} (best epoch {best['epoch']}, val F1 {best['f1']:.4f})")
            break
    info = dict(train_secs=time.time() - t0, history=hist, n_windows=len(W), steps=step,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2**30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    labels, tags, folds = D["labels"], D["tags"], D["folds"]
    if J.get("quick"):
        folds = {f: dict(list(v.items())[: J["quick_docs"]]) for f, v in folds.items()}
    method = J["method"]
    base_factory = lambda: load_backbone(J)
    log(f"arch={J.get('arch', 'txl')} method={method} attn={J['attn']} head={J['head']} device={device}")

    def get_teacher(which):
        tp = Path(J["kd_teacher_dir"]) / f"{which}_model.pt"
        assert tp.exists(), f"KD needs the teacher checkpoint {tp} (run the teacher job first)"
        tj = json.load(open(Path(J["kd_teacher_dir"]) / "job.json"))
        t = build_model(tj, base_factory, len(tags), tj["method"])
        t.load_state_dict(torch.load(tp, map_location="cpu"))
        t.to(device).eval()
        for p in t.parameters():
            p.requires_grad_(False)
        return t

    results = dict(name=J["name"], method=method, attn=J["attn"], head=J["head"], config=J)
    # ---- (1) model selection: train fold1 -> validate fold2
    model = build_model(J, base_factory, len(tags), method)
    results["params_total"], results["params_trainable"] = count_params(model)
    log(f"params total={results['params_total']/1e6:.2f}M trainable={results['params_trainable']/1e6:.3f}M")
    teacher = get_teacher("val") if method == "kd" else None
    if teacher is not None:
        results["teacher_params"] = count_params(teacher)[0]
    best, info = train_one(J, model, folds["fold1"], folds["fold2"], tags, labels, device, J["epochs"], log, teacher)
    results.update(val_macro_f1=best["f1"], best_epoch=best["epoch"], val_report=best["report"], val_train=info,
                   val_errors=error_breakdown({d: v["gold"] for d, v in folds["fold2"].items()}, best["pred"]))
    json.dump(best["pred"], open(out / "val_pred_fold2.json", "w"))
    log(f"VALIDATION fold1->fold2 strict macro-F1 = {best['f1']:.4f} (epoch {best['epoch']})")
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    del teacher
    # ---- (2) final model
    if J["final_fit"] == "all":
        epochs = max(1, best["epoch"] + J.get("final_extra_epochs", 0))
        model = build_model(J, base_factory, len(tags), method)
        teacher = get_teacher("final") if method == "kd" else None
        train_docs = {**folds["fold1"], **folds["fold2"]}
        _, finfo = train_one(J, model, train_docs, None, tags, labels, device, epochs, log, teacher)
        results["final_train"] = finfo
        del teacher
    else:
        model.load_state_dict(best["state"])
    model.to(device)
    if J["save_model"]:
        torch.save({n: p.detach().cpu() for n, p in model.state_dict().items()}, out / "final_model.pt")
    # ---- (3) test prediction
    t0 = time.time()
    test = folds["fold3"]
    lps = doc_logprobs(model, test, J["max_len"], J["eval_stride"], J["eval_batch_size"], device)
    pred = decode_docs(test, lps, tags)
    results["test_infer_secs"] = time.time() - t0
    all_ids = list(D["folds"]["fold3"].keys())
    final = {did: [[int(s), int(e), l] for s, e, l in pred.get(did, [])] for did in all_ids}
    assert set(final) == set(all_ids)
    for did, spans in final.items():
        n = len(D["folds"]["fold3"][did]["text"])
        assert all(0 <= s < e <= n and l in labels for s, e, l in spans), did
    pj = out / "prediction.json"
    json.dump(final, open(pj, "w"))
    zp = out / f"submission_{J['name']}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    results["test_entities"] = sum(len(v) for v in final.values())
    results["test_label_counts"] = dict(Counter(l for v in final.values() for _, _, l in v))
    json.dump(results, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> {zp} | test entities={results['test_entities']} | {json.dumps(results['test_label_counts'])}")


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p4/p2_ner.py


In [3]:
%%writefile /kaggle/working/p4/p4_cjpe.py
"""CS-781 Phase 4 - Court Judgment Prediction (CJPE: binary 0/1 per case, Macro-F1) on top of my Phase-1 models.

One file, used two ways (same as Phases 2-3):
  * imported by the notebook  -> data cleaning, tokenisation checks, chunk cache
  * run as a script            -> `python p4_cjpe.py job.json` trains / evaluates / predicts ONE experiment

Model (hierarchical: chunk encoder + document pooling)
  case text --(EXACT Phase-1 preprocessing + tokenizer)--> token ids
    --> keep the FIRST chunk (head: parties, court, what is appealed) and the LAST `n_tail` chunks (tail: arguments,
        reasoning - ILDC removed the final decision sentences) of `chunk_len` tokens; shorter cases are kept whole
    --> every chunk: Phase-1 backbone (bidirectional, + adaptation method) --> mean-pooled chunk vector
        + chunk-slot embedding (distance from the end) + head-flag embedding
    --> BiGRU over the chunks of the case --> attention pooling (+ max pooling) --> linear --> logits for {0, 1}
  Training samples `train_chunks` chunks per case (always including the LAST chunk) for speed; prediction uses all.
  The decision threshold on P(1) is tuned on the clean dev set (train is 59/41, dev and test are 50/50).

Data hygiene: dev.json of the release contains all 1517 test cases (with labels) and every real dev case twice,
and train.json contains ILDC-single inside ILDC-multi. load_cjpe() drops every dev/train row whose id or text is in
test, de-duplicates by text, and NEVER reads test labels. The clean dev set (994 cases) is used only for model
selection, threshold tuning and analysis - there is no train+dev refit in this phase.

Adaptation methods (method=...), Phase-4 list: full, partial, gradual (layer freezing/unfreezing), bitfit, lorma,
adalora, adapter, kd.  Plus `frozen` (frozen Phase-1 encoder, only the document head is trained: a probe).
  lorma    LoRMA_+ (Bihany, Patel & Modi 2025): y = (I + s*B A) W x                 (compulsory; KD teacher)
  kd       knowledge distillation: fine-tuned full-depth teacher -> shallower student (subset of Phase-1 layers),
           loss = a*CE + (1-a)*T^2*KL(teacher || student) on the same chunks       (compulsory)
  adalora  AdaLoRA via the `peft` library (SVD-form LoRA, importance-based rank re-allocation)
  bitfit   biases only (bias-free models get zero-initialised biases on every projection) + norm gains
  adapter  Houlsby bottleneck adapters on the attention-output and FFN-output projections
  partial  top `partial_top_layers` blocks (+ final norm) fine-tuned, the rest frozen
  gradual  gradual unfreezing (ULMFiT style): head only first, then the top 2/4/6 blocks are unfrozen
  full     full fine-tuning
"""
import hashlib
import json
import math
import multiprocessing as mp
import os
import pickle
import re
import sys
import time
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).resolve().parent))
import p2_ner as P2                            # noqa: E402  (Phase-2 loaders / encoders / LoRA / LoRMA / adapters)


# ============================================================== data: leak removal + de-duplication
def _md5(t):
    return hashlib.md5(t.encode("utf-8", "ignore")).hexdigest()


def _read(path):
    d = json.load(open(path, encoding="utf-8"))
    items = d if isinstance(d, list) else [dict(id=k, **v) if isinstance(v, dict) else dict(id=k, text=v) for k, v in d.items()]
    out = []
    for it in items:
        t = it.get("text", "")
        t = " ".join(t) if isinstance(t, list) else str(t)
        out.append(dict(id=str(it["id"]), text=t, label=it.get("label")))
    return out


def load_cjpe(data_dir, log=print):
    """-> dict(train, dev, test, stats). Every item: dict(id, text, label) (label None for test - never read)."""
    data_dir = Path(data_dir)
    test = [dict(id=x["id"], text=x["text"], label=None) for x in _read(data_dir / "Test" / "test.json")]
    test_ids = [x["id"] for x in test]
    assert len(set(test_ids)) == len(test_ids), "duplicate ids in test.json"
    test_ids, test_h = set(test_ids), {_md5(x["text"]) for x in test}

    def clean(items, banned_ids, banned_h):
        seen, out = {}, []
        st = Counter()
        for x in items:
            h = _md5(x["text"])
            if x["id"] in banned_ids or h in banned_h:
                st["dropped_overlap_with_test_or_dev"] += 1
                continue
            if h in seen:
                st["dropped_duplicate_text"] += 1
                st["duplicates_with_conflicting_label"] += int(seen[h] != x["label"])
                continue
            seen[h] = x["label"]
            assert x["label"] in (0, 1), f"label {x['label']!r} of {x['id']}"
            out.append(x)
        return out, dict(st)

    dev_raw = _read(data_dir / "Train" / "dev.json")
    dev, st_dev = clean(dev_raw, test_ids, test_h)
    del dev_raw
    ids = [x["id"] for x in dev]
    assert len(set(ids)) == len(ids), "dev still has duplicate ids after text de-duplication"
    train_raw = _read(data_dir / "Train" / "train.json")
    n_train_raw = len(train_raw)
    train, st_tr = clean(train_raw, test_ids | set(ids), test_h | {_md5(x["text"]) for x in dev})
    del train_raw
    stats = dict(train_rows=n_train_raw, train_kept=len(train), train=st_tr, train_labels=dict(Counter(x["label"] for x in train)),
                 dev_kept=len(dev), dev=st_dev, dev_labels=dict(Counter(x["label"] for x in dev)), test=len(test))
    log(json.dumps(stats))
    return dict(train=train, dev=dev, test=test, stats=stats)


# ============================================================== tokenisation (exact Phase-1 pipeline, head + tail)
_MAP = str.maketrans({
    "‘": "'", "’": "'", "‚": "'", "‛": "'", "“": '"', "”": '"',
    "„": '"', "«": '"', "»": '"', "–": "-", "—": "-", "−": "-",
    "‐": "-", "‑": "-", "‒": "-", "―": "-", " ": " ", "•": "*",
    "·": "*", "…": "...", "₹": "Rs. ",
})
_CTRL = re.compile(r"[\x00-\x08\x0e-\x1f\x7f]")


def txl_normalize(t):
    """Exactly prep_data.normalize_text of the Phase-1 Transformer-XL runs."""
    if not t:
        return ""
    if not t.isascii():
        t = t.translate(_MAP)
        t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode("ascii")
    t = _CTRL.sub("", t)
    return " ".join(t.split())


def text_fns(tok, arch):
    """(normalize(raw) -> the text Phase-1 pretraining tokenised,  encode(normalized text) -> ids)."""
    if tok.kind == "hf":                         # Archit-2-v2: " ".join(text.split()) -> ByteLevelBPETokenizer
        return (lambda t: " ".join(t.split())), (lambda s: tok.tok.encode(s).ids)
    if arch == "txl":                            # TXL: prep_data.normalize_text -> tokenizer.encode
        return txl_normalize, tok.tok.encode
    if tok.kind == "new":                        # new BPE: a1.normalize_text -> encode(normalize=False)
        return tok.mod.normalize_text, (lambda s: tok.tok.encode(s, normalize=False))
    if tok.norm == "clean":                      # RetNet: ret_data.clean_text -> ORIGINAL a1_tokenizer regex
        enc = tok.tok._encode_pretoken
        return P2.clean_text_ret, (lambda s: [i for pc in P2.PRE_RE_ORIG.findall(s) for i in enc(pc)])
    raise ValueError(f"no Phase-1 text pipeline for arch={arch} tokenizer={tok.kind}/{tok.norm}")


_WORD = re.compile(r"\S+")


def encode_head_tail(raw, norm_fn, enc_fn, head_tok, tail_tok):
    """-> (head_ids, tail_ids or None, n_words).  tail_ids is None when the WHOLE case was encoded (head_ids = all).
    Long cases: only the first ~head_tok+64 and last ~tail_tok+64 words are tokenised (every word gives >= 1 token).
    Exactness: the Phase-1 pre-tokenisers never let a token cross the start of a whitespace run, so encoding the
    text up to / from such a position gives exactly a prefix / suffix of the full encoding (asserted in the notebook)."""
    t = norm_fn(raw)
    spans = [m.span() for m in _WORD.finditer(t)]
    hw, tw = head_tok + 64, tail_tok + 64
    if len(spans) <= hw + tw:
        return np.asarray(enc_fn(t), np.int64), None, len(spans)
    head = enc_fn(t[:spans[hw - 1][1]])[:head_tok]
    tail = enc_fn(t[spans[len(spans) - tw - 1][1]:])
    tail = tail[-tail_tok:] if tail_tok else []
    return np.asarray(head, np.int64), np.asarray(tail, np.int64), len(spans)


def make_chunks(head, tail, n_head, n_tail, chunk_len, bos):
    """-> (list of uint16 arrays, truncated?)   every array = [<s>] + up to chunk_len-1 tokens (or chunk_len without <s>)."""
    c = chunk_len - (bos is not None)
    if tail is None and len(head) <= (n_head + n_tail) * c:
        ids = head if len(head) else np.asarray([P2_UNK_FALLBACK], np.int64)
        pieces, trunc = [ids[i:i + c] for i in range(0, len(ids), c)], False
    else:
        if tail is None:
            head, tail = head[:n_head * c], head[len(head) - n_tail * c:]
        head, tail = head[:n_head * c], tail[max(0, len(tail) - n_tail * c):]
        pieces = [head[i:i + c] for i in range(0, len(head), c)]
        pieces += [tail[max(0, j - c):j] for j in range(len(tail), 0, -c)][::-1]
        trunc = True
    if bos is not None:
        pieces = [np.concatenate([[bos], p]) for p in pieces]
    return [np.asarray(p, np.uint16) for p in pieces], trunc


P2_UNK_FALLBACK = 1          # a case with no tokens at all becomes one <unk>-like token (never happens on real data)
_W = {}                      # per-worker state (fork)


def _init_worker(spec):
    tok = P2.load_bpe(spec["tok_py"], spec["tok_path"], norm=spec["norm"])
    _W["norm"], _W["enc"] = text_fns(tok, spec["arch"])
    _W["spec"] = spec


def _work(i):
    s = _W["spec"]
    c = s["chunk_len"] - (s["bos"] is not None)
    h, t, nw = encode_head_tail(_W["texts"][i], _W["norm"], _W["enc"], s["n_head"] * c, s["n_tail"] * c)
    chunks, trunc = make_chunks(h, t, s["n_head"], s["n_tail"], s["chunk_len"], s["bos"])
    return i, chunks, trunc, nw


def prepare(data, spec, cache_path, n_proc=None, log=print):
    """Tokenise + chunk every split (multiprocessing, fork) and pickle {'spec', 'splits', 'stats'} to cache_path."""
    n_proc = n_proc or max(1, os.cpu_count() or 1)
    splits = {}
    for sp in ("train", "dev", "test"):
        docs = data[sp]
        _W["texts"] = [d["text"] for d in docs]
        out = [None] * len(docs)
        t0 = time.time()
        if n_proc > 1:
            with mp.get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(spec,)) as pool:
                for i, ch, tr, nw in pool.imap_unordered(_work, range(len(docs)), chunksize=32):
                    out[i] = (ch, tr, nw)
        else:
            _init_worker(spec)
            for i in range(len(docs)):
                out[i] = _work(i)[1:]
        splits[sp] = [dict(id=d["id"], y=d["label"], chunks=ch, trunc=tr, n_words=nw)
                      for d, (ch, tr, nw) in zip(docs, out)]
        log(f"[prepare] {sp}: {len(docs)} cases tokenised in {time.time() - t0:.0f}s")
    _W.pop("texts", None)
    pickle.dump(dict(spec=spec, splits=splits, stats=data.get("stats")), open(cache_path, "wb"), protocol=4)
    return splits


def chunk_stats(splits, spec):
    rows = {}
    for sp, docs in splits.items():
        nc = np.asarray([len(d["chunks"]) for d in docs])
        nt = np.asarray([sum(len(c) for c in d["chunks"]) for d in docs])
        nw = np.asarray([d["n_words"] for d in docs])
        rows[sp] = dict(cases=len(docs), words_p50=int(np.median(nw)), words_p90=int(np.percentile(nw, 90)),
                        chunks_mean=round(float(nc.mean()), 2), chunks_max=int(nc.max()), tokens_kept_p50=int(np.median(nt)),
                        truncated_pct=round(100 * float(np.mean([d["trunc"] for d in docs])), 1),
                        labels=str(dict(Counter(d["y"] for d in docs))) if docs and docs[0]["y"] is not None else "-")
    return rows


# ============================================================== metrics
def bin_metrics(y, p, thr):
    y = np.asarray(y).astype(int)
    pred = (np.asarray(p) >= thr).astype(int)
    out = dict(thr=float(thr))
    f1s = []
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum()); fn = int(((pred != c) & (y == c)).sum())
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * P * R / (P + R) if P + R else 0.0
        out[f"p{c}"], out[f"r{c}"], out[f"f1_{c}"] = P, R, f
        f1s.append(f)
    out["macro_f1"] = float(np.mean(f1s))
    out["macro_p"] = float((out["p0"] + out["p1"]) / 2)
    out["macro_r"] = float((out["r0"] + out["r1"]) / 2)
    out["accuracy"] = float((pred == y).mean()) if len(y) else 0.0
    out["confusion"] = [[int(((y == 0) & (pred == 0)).sum()), int(((y == 0) & (pred == 1)).sum())],
                        [int(((y == 1) & (pred == 0)).sum()), int(((y == 1) & (pred == 1)).sum())]]   # rows = gold 0/1
    return out


def tune_threshold(y, p):
    """Threshold on P(1) maximising Macro-F1 (grid 0.05..0.95; ties -> closest to 0.5)."""
    best = None
    for t in np.round(np.arange(0.05, 0.9501, 0.01), 2):
        f = bin_metrics(y, p, t)["macro_f1"]
        key = (round(f, 6), -abs(t - 0.5))
        if best is None or key > best[0]:
            best = (key, float(t), f)
    return best[1], best[2]


def roc_auc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    if not n1 or not n0:
        return float("nan")
    r = np.empty(len(p)); r[np.argsort(p, kind="mergesort")] = np.arange(1, len(p) + 1)
    for v in np.unique(p):                       # average ranks of ties
        m = p == v
        if m.sum() > 1:
            r[m] = r[m].mean()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def length_buckets(docs, y, p, thr):
    out = {}
    for name, lo, hi in [("<1k words", 0, 1000), ("1k-3k", 1000, 3000), ("3k-8k", 3000, 8000), (">8k", 8000, 10 ** 9)]:
        idx = [i for i, d in enumerate(docs) if lo <= d["n_words"] < hi]
        if idx:
            m = bin_metrics(np.asarray(y)[idx], np.asarray(p)[idx], thr)
            out[name] = dict(n=len(idx), macro_f1=round(m["macro_f1"], 4), acc=round(m["accuracy"], 4))
    return out


# ============================================================== backbone wrappers
class EncTaggerBOS(P2.EncTagger):
    """Encoder-decoder v2 used BART-style, fed like its Phase-1 pretraining: encoder <- chunk,
    decoder <- [<s>] + chunk; returns the decoder states of the chunk tokens (the <s> state is dropped)."""

    def __init__(self, *a, bos_id=1, **kw):
        super().__init__(*a, **kw)
        self.bos_id = bos_id
        assert self.use_decoder

    def hidden(self, ids, attn_mask):
        mem = self.encode(ids, attn_mask)
        B, dev = ids.size(0), ids.device
        dids = torch.cat([torch.full((B, 1), self.bos_id, dtype=ids.dtype, device=dev), ids], 1)
        dam = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=dev), attn_mask], 1)
        T = dids.size(1)
        x = self.emb_drop(self.tok(dids) + self.dec_pos(torch.arange(T, device=dev))[None])
        cmask = attn_mask[:, None, None, :]
        smask = dam[:, None, None, :]
        if self.attn_mode == "causal":
            smask = smask & torch.ones(T, T, dtype=torch.bool, device=dev).tril()[None, None]
        smask = smask | torch.eye(T, dtype=torch.bool, device=dev)[None, None]
        for l in self.dec_blocks:
            x = x + l.drop(l.self_attn(l.n1(x), mask=smask))
            x = x + l.drop(l.cross(l.n2(x), mem=mem, mask=cmask))
            x = x + l.drop(l.ffn(l.n3(x)))
        return self.dec_norm(x)[:, 1:]


def build_encoder(J, layers=None):
    """Phase-1 backbone wrapped by the (already verified) Phase-2 encoder class; only .hidden() is used."""
    base = P2.load_backbone(J)
    arch = J["arch"]
    cls = {"dec": P2.DecTagger, "encdec": P2.EncTagger, "encdec2": P2.ED2Tagger, "ret": P2.RetTagger}.get(arch, P2.TXLTagger)
    kw = dict(use_decoder=J.get("encdec_use_decoder", False)) if cls in (P2.EncTagger, P2.ED2Tagger) else {}
    if cls is P2.EncTagger and J.get("dec_bos") and kw["use_decoder"]:
        cls, kw = EncTaggerBOS, dict(kw, bos_id=J.get("bos_id", 1))
    enc = cls(base, 2, attn=J["attn"], layers=layers, dropout=J["dropout"], head="linear", **kw)
    enc.head = nn.Identity()                   # the Phase-2 token head is not used
    return enc


# ============================================================== document model
class CJPEModel(nn.Module):
    def __init__(self, enc, J):
        super().__init__()
        self.enc = enc
        d = enc.d
        self.slot = nn.Embedding(16, d)          # distance of the chunk from the last stored chunk (0 = last)
        self.hflag = nn.Embedding(2, d)          # 1 = first chunk of the case (case header)
        nn.init.normal_(self.slot.weight, 0, 0.02); nn.init.normal_(self.hflag.weight, 0, 0.02)
        self.drop = nn.Dropout(J["dropout"])
        self.doc_pool = J["doc_pool"]
        if self.doc_pool == "bigru":
            self.rnn = nn.GRU(d, J["rnn_hidden"], batch_first=True, bidirectional=True)
            dout = 2 * J["rnn_hidden"]
        else:                                    # 'attn': attention/max pooling straight over the chunk vectors
            self.rnn, dout = None, d
        self.att = nn.Sequential(nn.Linear(dout, 128), nn.Tanh(), nn.Linear(128, 1))
        self.cls = nn.Linear(2 * dout, 2)
        self._amp, self._ckpt, self._ckpt_micro, self._n_overflow = True, bool(J.get("grad_ckpt")), J.get("ckpt_micro", 4), 0

    def chunk_vecs(self, ids, am):
        h = self.enc.hidden(ids, am)
        m = am.unsqueeze(-1).to(h.dtype)
        return (h * m).sum(1) / m.sum(1).clamp(min=1.0)

    def pool(self, vecs, slots, hflags, lengths):
        """vecs [N, d] = the chunks of B cases, case after case; lengths = chunks per case -> logits [B, 2]."""
        with torch.autocast(device_type=vecs.device.type, enabled=False):
            x = self.drop(vecs.float() + self.slot(slots).float() + self.hflag(hflags).float())
            X = nn.utils.rnn.pad_sequence(list(torch.split(x, lengths)), batch_first=True)
            M = torch.zeros(X.shape[:2], dtype=torch.bool, device=x.device)
            for b, n in enumerate(lengths):
                M[b, :n] = True
            if self.rnn is not None:
                pk = nn.utils.rnn.pack_padded_sequence(X, torch.as_tensor(lengths, dtype=torch.long), batch_first=True,
                                                       enforce_sorted=False)
                H, _ = self.rnn(pk)
                H, _ = nn.utils.rnn.pad_packed_sequence(H, batch_first=True, total_length=X.size(1))
            else:
                H = X
            H = self.drop(H)
            a = torch.softmax(self.att(H).squeeze(-1).masked_fill(~M, -1e4), -1)
            att = (a.unsqueeze(-1) * H).sum(1)
            mx = H.masked_fill(~M.unsqueeze(-1), -1e4).max(1).values
            return self.cls(torch.cat([att, mx], -1))


def encode_chunks(model, ids, am, log=None):
    """Chunk vectors [N, d] in fp16 autocast; if fp16 overflows (inf/NaN), recompute in fp32 (after 3 overflows the
    model stays in fp32). Optional gradient checkpointing per `ckpt_micro` chunks (enabled automatically on OOM)."""
    def fwd():
        if model._ckpt and torch.is_grad_enabled():
            m, outs = model._ckpt_micro, []
            for k in range(0, ids.size(0), m):
                outs.append(torch.utils.checkpoint.checkpoint(model.chunk_vecs, ids[k:k + m], am[k:k + m], use_reentrant=False))
            return torch.cat(outs)
        return model.chunk_vecs(ids, am)

    if ids.is_cuda and model._amp:
        with torch.autocast("cuda", dtype=torch.float16):
            v = fwd()
        if torch.isfinite(v).all():
            return v.float()
        model._n_overflow += 1
        if model._n_overflow >= 3:
            model._amp = False
            if log:
                log("fp16 overflow in the encoder 3x -> encoder runs in fp32 from now on")
    return fwd().float()


def collate_chunks(arrs):
    L = max(len(a) for a in arrs)
    ids = torch.zeros(len(arrs), L, dtype=torch.long)
    am = torch.zeros(len(arrs), L, dtype=torch.bool)
    for i, a in enumerate(arrs):
        ids[i, :len(a)] = torch.from_numpy(a.astype(np.int64))
        am[i, :len(a)] = True
    return ids, am


def make_batch(docs, sels, device):
    arrs, slots, hfl, lengths = [], [], [], []
    for d, sel in zip(docs, sels):
        n = len(d["chunks"])
        for k in sel:
            arrs.append(d["chunks"][k]); slots.append(min(15, n - 1 - k)); hfl.append(int(k == 0))
        lengths.append(len(sel))
    ids, am = collate_chunks(arrs)
    t = lambda v: torch.as_tensor(v, dtype=torch.long, device=device)
    return ids.to(device), am.to(device), t(slots), t(hfl), lengths


def forward_docs(model, docs, sels, device, log=None, no_grad_enc=False):
    ids, am, slots, hfl, lengths = make_batch(docs, sels, device)
    if no_grad_enc:
        with torch.no_grad():
            v = encode_chunks(model, ids, am, log)
    else:
        v = encode_chunks(model, ids, am, log)
    return model.pool(v, slots, hfl, lengths), int(am.sum())


SECTIONS = {                                     # chunk subsets for the section ablation (indices into stored chunks)
    "all": lambda n: list(range(n)),
    "last1": lambda n: [n - 1],
    "last2": lambda n: list(range(max(0, n - 2), n)),
    "last4": lambda n: list(range(max(0, n - 4), n)),
    "head1": lambda n: [0],
    "head1+last1": lambda n: sorted({0, n - 1}),
}


@torch.no_grad()
def predict_probs(model, docs, device, chunk_bs, section="all"):
    """-> P(label = 1) for every case (numpy, in order)."""
    model.eval()
    sel_fn = SECTIONS[section]
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["chunks"]))
    probs = np.zeros(len(docs), np.float32)
    k = 0
    while k < len(order):
        batch, n = [], 0
        while k < len(order) and (not batch or n + len(sel_fn(len(docs[order[k]]["chunks"]))) <= chunk_bs):
            batch.append(order[k]); n += len(sel_fn(len(docs[order[k]]["chunks"]))); k += 1
        bd = [docs[i] for i in batch]
        lg, _ = forward_docs(model, bd, [sel_fn(len(d["chunks"])) for d in bd], device)
        probs[batch] = F.softmax(lg.float(), -1)[:, 1].cpu().numpy()
    return probs


# ============================================================== adaptation methods
def _slots(enc, names):
    out = []
    for blk in enc.blocks:
        out += P2._linear_slots(blk, names)
    return out


def apply_method(model, method, J):
    """Freeze / wrap the encoder for the adaptation method. Returns (new PEFT parameters, body parameters that
    gradual unfreezing will switch on later)."""
    enc = model.enc
    for p in enc.parameters():
        p.requires_grad_(method in ("full", "kd"))
    new, later = [], []
    if method in ("lora", "lorma"):
        sl = _slots(enc, J["lowrank_targets"])
        assert sl, "no LoRA/LoRMA target found - check lowrank_targets"
        for holder, nm in sl:
            m = P2.LowRankLinear(getattr(holder, nm), J["rank"], J["alpha"], J["lowrank_dropout"], method)
            setattr(holder, nm, m)
            new += [m.A, m.B]
    elif method == "adapter":
        sl = _slots(enc, J["adapter_targets"])
        assert sl, "no adapter target found - check adapter_targets"
        for holder, nm in sl:
            m = P2.Adapter(getattr(holder, nm), J["adapter_dim"])
            setattr(holder, nm, m)
            new += list(m.down.parameters()) + list(m.up.parameters())
    elif method == "bitfit":
        if J["arch"] in ("dec", "ret"):          # bias-free projections: add zero-initialised trainable biases
            for holder, nm in _slots(enc, J["lowrank_targets"]):
                setattr(holder, nm, P2.BiasLinear(getattr(holder, nm)))
        for n, p in enc.named_parameters():
            if n.endswith("bias") or n.endswith(".u") or n.endswith(".v") or n.endswith(".gn_w") or n.endswith(".gn_b"):
                p.requires_grad_(True)
        for m in enc.modules():
            if "Norm" in type(m).__name__:
                for p in m.parameters():
                    p.requires_grad_(True)
        new = [p for p in enc.parameters() if p.requires_grad]
    elif method == "adalora":
        from peft import AdaLoraConfig
        from peft.tuners.adalora import AdaLoraModel
        total = J["_total_steps"]
        cfg = AdaLoraConfig(target_modules=J["lowrank_targets"], init_r=J["ada_init_r"], target_r=J["ada_target_r"],
                            lora_alpha=J["alpha"], lora_dropout=J["lowrank_dropout"], total_step=total,
                            tinit=int(0.1 * total), tfinal=int(0.2 * total), deltaT=max(1, int(0.02 * total)),
                            orth_reg_weight=J["ada_orth_reg"])
        ada = AdaLoraModel(enc, cfg, "default")           # injects the adapters into enc IN PLACE
        object.__setattr__(model, "_ada", ada)             # a handle (not a sub-module) for update_and_allocate
        for n, p in enc.named_parameters():
            if "lora_" in n:
                p.requires_grad_(True)
                new.append(p)
        assert new, "AdaLoRA found no target modules"
    elif method == "partial":
        for blk in list(enc.blocks)[-J["partial_top_layers"]:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in enc.final_norm.parameters():
            p.requires_grad_(True)
    elif method == "gradual":
        later = [p for p in enc.parameters()]             # all frozen now; unfrozen top-down during training
    elif method in ("frozen", "full", "kd"):
        pass
    else:
        raise ValueError(method)
    return new, later


def set_gradual_stage(model, n_top):
    enc = model.enc
    blocks = list(enc.blocks)
    for i, blk in enumerate(blocks):
        on = i >= len(blocks) - n_top
        for p in blk.parameters():
            p.requires_grad_(on)
    for p in enc.final_norm.parameters():
        p.requires_grad_(n_top > 0)


def adalora_orth_reg(model):
    reg, n = 0.0, 0
    for name, p in model.enc.named_parameters():
        if ("lora_A" in name or "lora_B" in name) and p.requires_grad and p.dim() == 2:
            pc = p @ p.T if "lora_A" in name else p.T @ p
            reg = reg + torch.norm(pc - torch.eye(pc.size(0), device=p.device, dtype=pc.dtype), p="fro")
            n += 1
    return reg / max(n, 1)


def build_model(J, method):
    layers = J["kd_student_layers"] if method == "kd" else None
    enc = build_encoder(J, layers)
    p1_ids = {id(p) for p in enc.parameters()}            # Phase-1 weights (frozen ones are not saved per job)
    model = CJPEModel(enc, J)
    new, later = apply_method(model, method, J)
    model._p1_ids = p1_ids
    return model, new, later


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================== training
def pick_chunks(n, k, rng):
    """k chunks of a case with n stored chunks: always the last one, the rest at random, in document order."""
    if n <= k:
        return list(range(n))
    return sorted(rng.choice(n - 1, size=k - 1, replace=False).tolist()) + [n - 1]


def class_weights(docs, mode):
    if mode == "none":
        return None
    cnt = np.bincount([d["y"] for d in docs], minlength=2).astype(np.float64)
    return torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32)


def train_one(J, model, new_params, later_params, train_docs, dev_docs, device, log, teacher=None):
    torch.manual_seed(J["seed"])
    rng = np.random.default_rng(J["seed"])
    model.to(device)
    cw = class_weights(train_docs, J["class_weight"])
    cw = cw.to(device) if cw is not None else None
    new_ids = {id(p) for p in new_params}
    head = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    peft_p = [p for p in model.enc.parameters() if id(p) in new_ids]
    body = [p for p in model.enc.parameters() if id(p) not in new_ids and (p.requires_grad or any(p is q for q in later_params))]
    groups = [dict(params=head, lr=J["head_lr"], weight_decay=J["weight_decay"])]
    if peft_p:
        groups.append(dict(params=peft_p, lr=J["peft_lr"], weight_decay=0.0 if J["method"] == "bitfit" else J["weight_decay"]))
    if body:
        groups.append(dict(params=body, lr=J["lr"], weight_decay=J["weight_decay"]))
    opt = torch.optim.AdamW(groups)
    for g in opt.param_groups:
        g["base_lr"] = g["lr"]
    opt_ids = {id(p) for g in opt.param_groups for p in g["params"]}
    frozen_ids = {i for i in model._p1_ids if i not in opt_ids}          # Phase-1 weights this method never trains
    keep_keys = ({n for n, p in model.named_parameters(remove_duplicate=False) if id(p) not in frozen_ids}
                 | {n for n, _ in model.named_buffers()})                   # everything else (incl. AdaLoRA ranknum)
    model._keep_keys = keep_keys
    enc_trainable = any(p.requires_grad for p in model.enc.parameters()) or bool(later_params)

    B, K = J["docs_per_step"], J["train_chunks"]
    steps_per_epoch = math.ceil(len(train_docs) / B)
    total = max(10, int(J["epochs"] * steps_per_epoch))
    planned = total
    warm = max(1, int(J["warmup"] * total))
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 10, enabled=use_amp)
    ada = getattr(model, "_ada", None)
    if use_amp:
        torch.cuda.reset_peak_memory_stats(device)
    n_dev_chunks = sum(len(d["chunks"]) for d in dev_docs)
    best = dict(f1=-1.0, step=0, state=None, probs=None, thr=0.5)
    hist, evals = [], []
    step, n_nan, n_skip, n_oom, tok_seen, docs_seen = 0, 0, 0, 0, 0, 0
    budget = J["time_budget_h"] * 3600
    t0 = time.time()
    t_cal = None
    next_eval = [int(total * (i + 1) / J["n_evals"]) for i in range(J["n_evals"])]
    gstage = -1

    def lr_factor(s):
        return (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))

    def do_eval(tag):
        te = time.time()
        probs = predict_probs(model, dev_docs, device, J["eval_chunk_bs"])
        y = [d["y"] for d in dev_docs]
        thr, f1t = tune_threshold(y, probs)
        m05 = bin_metrics(y, probs, 0.5)
        rec = dict(step=step, docs_seen=docs_seen, epoch=round(docs_seen / len(train_docs), 3), dev_f1_tuned=round(f1t, 4),
                   thr=thr, dev_f1_05=round(m05["macro_f1"], 4), dev_auc=round(roc_auc(y, probs), 4),
                   eval_secs=round(time.time() - te, 1), tag=tag)
        evals.append(rec)
        log("[EVAL] " + json.dumps(rec))
        if f1t > best["f1"]:
            best.update(f1=f1t, step=step, probs=probs, thr=thr,
                        state={k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep_keys})
        model.train()
        return time.time() - te

    order = []
    model.train()
    tl, tn = 0.0, 0
    while step < total:
        if not order:
            order = rng.permutation(len(train_docs)).tolist()
        idx = [order.pop() for _ in range(min(B, len(order)))]
        docs = [train_docs[i] for i in idx]
        if J["method"] == "gradual":                       # gradual unfreezing: (fraction of training, top blocks)
            st = max(i for i, (f, _) in enumerate(J["gradual_schedule"]) if step >= f * total)
            if st != gstage:
                gstage = st
                set_gradual_stage(model, J["gradual_schedule"][st][1])
                log(f"gradual unfreezing: step {step} -> top {J['gradual_schedule'][st][1]} blocks trainable")
        f = lr_factor(step)
        for g in opt.param_groups:
            g["lr"] = g["base_lr"] * f
        y = torch.as_tensor([d["y"] for d in docs], device=device)
        while True:
            try:
                sels = [pick_chunks(len(d["chunks"]), K, rng) for d in docs]
                lg, ntok = forward_docs(model, docs, sels, device, log, no_grad_enc=not enc_trainable)
                lg = lg.float()
                loss = F.cross_entropy(lg, y, weight=cw)
                if teacher is not None:
                    with torch.no_grad():
                        tlg, _ = forward_docs(teacher, docs, sels, device)
                    T = J["kd_temperature"]
                    kl = F.kl_div(F.log_softmax(lg / T, -1), F.log_softmax(tlg.float() / T, -1), log_target=True, reduction="batchmean")
                    loss = J["kd_alpha"] * loss + (1 - J["kd_alpha"]) * T * T * kl
                if ada is not None and J["ada_orth_reg"] > 0:
                    loss = loss + J["ada_orth_reg"] * adalora_orth_reg(model)
                if not torch.isfinite(loss):
                    n_nan += 1
                    if n_nan <= 5 or n_nan % 100 == 0:
                        log(f"non-finite loss at step {step} (#{n_nan}) - batch skipped")
                    opt.zero_grad(set_to_none=True)
                    break
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                for g in opt.param_groups:
                    torch.nn.utils.clip_grad_norm_([p for p in g["params"] if p.grad is not None], 1.0)
                sc0 = scaler.get_scale() if use_amp else 1.0
                scaler.step(opt)
                scaler.update()
                if use_amp and scaler.get_scale() < sc0:
                    n_skip += 1
                if ada is not None:
                    ada.update_and_allocate(step)
                tl += loss.item(); tn += 1; tok_seen += ntok
                break
            except torch.cuda.OutOfMemoryError:
                n_oom += 1
                opt.zero_grad(set_to_none=True)
                lg = loss = None
                torch.cuda.empty_cache()
                if not model._ckpt:
                    model._ckpt = True
                    log(f"CUDA OOM at step {step} -> gradient checkpointing ON (every {model._ckpt_micro} chunks)")
                elif K > 2:
                    K -= 1
                    log(f"CUDA OOM again -> train_chunks reduced to {K}")
                else:
                    raise
        step += 1
        docs_seen += len(docs)
        # ---- time calibration: fit the number of steps into the time budget (the LR decay ends exactly there)
        if step == 10:
            t_cal = time.time()
        if step == J["calib_steps"] and t_cal is not None:
            sps = (time.time() - t_cal) / (step - 10)
            cps_train = (B * min(K, 8)) / sps                         # chunks/s while training (upper bound on chunks)
            speed = 2.5 if enc_trainable else 1.0                     # forward-only inference is ~2.5-3x faster
            eval_s = n_dev_chunks / (speed * cps_train)
            test_s = J.get("_n_test_chunks", n_dev_chunks) / (speed * cps_train)
            reserve = (J["n_evals"] + (3 if J.get("section_ablation") else 1)) * eval_s + test_s + 120
            fit = step + int((budget - (time.time() - t0) - reserve) / sps)
            total = max(step + 20, min(planned, fit))
            warm = min(warm, max(1, int(J["warmup"] * total)))
            next_eval = [max(step + 1, int(total * (i + 1) / J["n_evals"])) for i in range(J["n_evals"])]
            if ada is not None:                                       # keep AdaLoRA's rank schedule consistent
                c = ada.peft_config["default"]
                c.total_step, c.tinit, c.tfinal, c.deltaT = total, int(0.1 * total), int(0.2 * total), max(1, int(0.02 * total))
            log(f"calibration: {sps:.3f} s/step ({B} cases x <= {K} chunks) | ~{eval_s / 60:.1f} min per dev eval | "
                f"steps {planned} planned -> {total} fit the {J['time_budget_h']:.2f} h budget "
                f"(= {total * B / len(train_docs):.2f} epochs of {len(train_docs)} cases)")
        if step % J["log_every"] == 0:
            el = time.time() - t0
            log(json.dumps(dict(step=step, total=total, loss=round(tl / max(tn, 1), 4), lr_f=round(f, 4),
                                tok_s=int(tok_seen / max(el, 1e-6)), min=round(el / 60, 1), nan=n_nan, skip=n_skip,
                                fp32=not model._amp, ckpt=model._ckpt)))
            hist.append(dict(step=step, loss=tl / max(tn, 1), min=el / 60))
            tl, tn = 0.0, 0
        if step in next_eval or step >= total:
            do_eval("scheduled")
            next_eval = [s for s in next_eval if s > step]
        elif time.time() - t0 > 1.08 * budget:
            log(f"time budget exceeded at step {step}/{total} -> final eval and stop")
            do_eval("time-stop")
            break
    info = dict(train_secs=time.time() - t0, steps=step, planned_steps=planned, docs_seen=docs_seen,
                epochs=docs_seen / len(train_docs), tok_per_s=tok_seen / max(time.time() - t0, 1e-6),
                nan_batches=n_nan, skipped_steps=n_skip, ooms=n_oom, fp32=not model._amp, grad_ckpt=model._ckpt,
                train_chunks_final=K, history=hist, evals=evals,
                peak_mem_gb=(torch.cuda.max_memory_allocated(device) / 2 ** 30) if use_amp else None)
    return best, info


# ============================================================== one experiment (script entry point)
def write_zip(pred, path):
    path = Path(path)
    pj = path.with_suffix(".json")
    json.dump(pred, open(pj, "w", encoding="utf-8"))
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    os.remove(pj)
    return path


def check_prediction(pred, test_docs):
    assert isinstance(pred, list) and len(pred) == len(test_docs), "one prediction per test case"
    assert [p["id"] for p in pred] == [d["id"] for d in test_docs], "ids must be exactly the test ids (same order)"
    assert all(type(p["pred"]) is int and p["pred"] in (0, 1) for p in pred), "pred must be the integer 0 or 1"


def subsample(docs, n, seed):
    if n is None or len(docs) <= n:
        return docs
    idx = np.random.default_rng(seed).choice(len(docs), n, replace=False)
    return [docs[i] for i in sorted(idx)]


def run_job(job_path):
    J = json.load(open(job_path))
    out = Path(J["out_dir"]); out.mkdir(parents=True, exist_ok=True)
    logf = open(out / "log.txt", "a")

    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}][{J['name']}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    D = pickle.load(open(J["cache"], "rb"))
    S = D["splits"]
    train, dev, test = S["train"], S["dev"], S["test"]
    if J.get("quick"):
        q = J["quick_docs"]
        train, dev, test = subsample(train, q, 1), subsample(dev, max(20, q // 2), 2), subsample(test, max(20, q // 2), 3)
    J["_n_test_chunks"] = sum(len(d["chunks"]) for d in test)
    method = J["method"]
    log(f"arch={J['arch']} method={method} doc_pool={J['doc_pool']} attn={J['attn']} device={device} | "
        f"train {len(train)} dev {len(dev)} test {len(test)} cases | budget {J['time_budget_h']:.2f} h")
    J["_total_steps"] = max(10, int(J["epochs"] * math.ceil(len(train) / J["docs_per_step"])))
    R = dict(name=J["name"], method=method, doc_pool=J["doc_pool"], config={k: v for k, v in J.items() if not k.startswith("_")})

    teacher = None
    if method == "kd":
        tdir = Path(J["kd_teacher_dir"])
        tp = tdir / "val_model.pt"
        assert tp.exists(), f"KD needs the teacher checkpoint {tp} (run the teacher job first, with save_model=True)"
        tj = json.load(open(tdir / "job.json"))
        tj["_total_steps"] = 1000
        teacher, _, _ = build_model(tj, tj["method"])
        missing, unexpected = teacher.load_state_dict(torch.load(tp, map_location="cpu"), strict=False)
        assert not unexpected, f"teacher checkpoint has unexpected keys {unexpected[:5]}"
        teacher.to(device).eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
        tres = json.load(open(tdir / "results.json")) if (tdir / "results.json").exists() else {}
        R["teacher"] = dict(name=tj["name"], method=tj["method"], params=count_params(teacher)[0],
                            dev_f1=tres.get("dev", {}).get("macro_f1"))
        log(f"teacher {tj['name']} ({tj['method']}) {R['teacher']['params'] / 1e6:.2f}M params, dev F1 {R['teacher']['dev_f1']}")

    model, new, later = build_model(J, method)
    R["params_total"] = count_params(model)[0]
    R["params_trainable"] = sum(p.numel() for p in model.parameters() if p.requires_grad or any(p is q for q in later))
    R["params_encoder"] = sum(p.numel() for p in model.enc.parameters())
    log(f"params total={R['params_total'] / 1e6:.2f}M (encoder {R['params_encoder'] / 1e6:.2f}M) "
        f"trainable={R['params_trainable'] / 1e6:.3f}M | PEFT tensors {len(new)}")

    best, info = train_one(J, model, new, later, train, dev, device, log, teacher)
    del teacher
    R["train"] = info
    model.load_state_dict(best["state"], strict=False)
    model.to(device).eval()
    if J["save_model"]:
        torch.save(best["state"], out / "val_model.pt")
    if method == "gradual":                        # report what was trainable at the end of gradual unfreezing
        R["params_trainable"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    y = [d["y"] for d in dev]
    yy = np.asarray(y)
    # dev predictions with different chunk subsets: 'all' (default) vs 'last4' (closer to the 4-chunk training samples);
    # the better one on dev (tuned threshold each) is used for test. Full set of subsets = section ablation (LoRMA job).
    infer = J.get("infer_sections", ["all", "last4"])
    secs = list(SECTIONS) if J.get("section_ablation") else infer
    cand = {"all": (best["probs"], best["thr"])}
    for sec in secs:
        if sec not in cand:
            ps = predict_probs(model, dev, device, J["eval_chunk_bs"], sec)
            cand[sec] = (ps, tune_threshold(y, ps)[0])
    if J.get("section_ablation"):                  # which part of the judgment carries the outcome signal?
        R["sections"] = {sec: dict(f1_at_model_thr=round(bin_metrics(y, ps, best["thr"])["macro_f1"], 4),
                                   f1_retuned=round(bin_metrics(y, ps, t)["macro_f1"], 4), thr_retuned=t,
                                   auc=round(roc_auc(y, ps), 4)) for sec, (ps, t) in cand.items()}
        log("SECTION ABLATION (dev) " + json.dumps(R["sections"]))
    pick = max(infer, key=lambda s: (round(bin_metrics(y, cand[s][0], cand[s][1])["macro_f1"], 6), s == "all"))
    p, thr = cand[pick]
    R["infer_section"] = pick
    R["dev_f1_by_infer_section"] = {s: round(bin_metrics(y, cand[s][0], cand[s][1])["macro_f1"], 4) for s in infer}
    R["dev"] = bin_metrics(y, p, thr)
    R["dev_at_0.5"] = bin_metrics(y, p, 0.5)
    R["dev_auc"] = roc_auc(y, p)
    R["best_step"] = best["step"]
    R["dev_by_length"] = length_buckets(dev, y, p, thr)
    R["dev_errors"] = dict(false_pos=[dev[i]["id"] for i in np.where((p >= thr) & (yy == 0))[0][:50]],
                           false_neg=[dev[i]["id"] for i in np.where((p < thr) & (yy == 1))[0][:50]])
    np.savez_compressed(out / "dev_probs.npz", ids=np.asarray([d["id"] for d in dev]), probs=p, y=yy)
    log(f"DEV macro-F1 {R['dev']['macro_f1']:.4f} @ threshold {thr:.2f} with chunks='{pick}' "
        f"({R['dev_f1_by_infer_section']}) | at 0.5: {R['dev_at_0.5']['macro_f1']:.4f} | AUC {R['dev_auc']:.4f} | "
        f"best step {best['step']} | by length {json.dumps(R['dev_by_length'])}")

    t0 = time.time()
    probs = predict_probs(model, test, device, J["eval_chunk_bs"], pick)
    R["test_infer_secs"] = time.time() - t0
    R["test_infer_ms_per_case"] = 1000 * R["test_infer_secs"] / max(1, len(test))
    np.savez_compressed(out / "test_probs.npz", ids=np.asarray([d["id"] for d in test]), probs=probs)
    pred = [dict(id=d["id"], pred=int(pv >= thr)) for d, pv in zip(test, probs)]
    check_prediction(pred, test)
    write_zip(pred, out / f"submission_{J['name']}.zip")
    cnt = Counter(x["pred"] for x in pred)
    R["test_label_counts"] = {str(k): v for k, v in sorted(cnt.items())}
    R["collapsed"] = min(cnt.get(0, 0), cnt.get(1, 0)) < 0.05 * len(pred)
    json.dump(R, open(out / "results.json", "w"), indent=1, default=str)
    log(f"DONE -> submission_{J['name']}.zip | test predictions {R['test_label_counts']}"
        + (" | !!! COLLAPSED (one class) - do NOT upload" if R["collapsed"] else ""))


if __name__ == "__main__":
    run_job(sys.argv[1])


Writing /kaggle/working/p4/p4_cjpe.py


In [4]:
# ============================================================ 2. DATA + TOKENISATION CHECKS + CHUNK CACHE + MODEL CHECK
sys.path.insert(0, str(WORK))
import importlib, p2_ner as P2, p4_cjpe as P4; importlib.reload(P2); importlib.reload(P4)
import torch, numpy as np, pandas as pd

# (a) data: drop the dev rows that are test cases, de-duplicate, never read test labels
data = P4.load_cjpe(DATA)
st = data["stats"]
assert not ({d["id"] for d in data["dev"]} & {d["id"] for d in data["test"]}), "dev/test overlap left"
assert not ({d["id"] for d in data["train"]} & {d["id"] for d in data["test"]}), "train/test overlap left"
print(f"train {st['train_rows']} rows -> {st['train_kept']} unique cases {st['train_labels']} | dev -> {st['dev_kept']} clean cases "
      f"{st['dev_labels']} | test {st['test']} cases (labels never used)")

# (b) token ids must be identical to the Phase-1 pipeline
tok = P2.load_bpe(TOK_PY, TOK_PATH, norm=NORM)
print("tokenizer kind:", tok.kind, "| normalisation:", NORM, "| vocab", tok.vocab_size); assert tok.vocab_size == 32000
NORM_FN, ENC_FN = P4.text_fns(tok, ARCH)
sample = [d["text"][:6000] for d in data["dev"][:4] + data["train"][:4]]
if ARCH == "ret":                          # Phase-1 RetNet: ret_data.clean_text -> ORIGINAL a1_tokenizer.encode
    if r'?[^\s\w]+|\s+")' in open(TOK_PY).read():         # attached a1_tokenizer.py is the original (unpatched) one
        ref = lambda s: tok.tok.encode(P2.clean_text_ret(s)); print("checking against your original a1_tokenizer.encode")
    else:
        ref = lambda s: [i for pc in P2.PRE_RE_ORIG.findall(P2.clean_text_ret(s)) for i in tok.tok._encode_pretoken(pc)]
        print("attached a1_tokenizer.py has the underscore fix; Phase-1 RetNet used the ORIGINAL regex, which p2_ner applies")
else:
    ref = tok.reference_encode
bad = [s for s in sample if list(ENC_FN(NORM_FN(s))) != list(ref(s))]
print(f"token-id check on {len(sample)} case texts: {len(sample) - len(bad)} identical"); assert not bad

# (c) head/tail encoding == exact prefix/suffix of the full encoding (small budgets so the partial path is exercised)
longs = [d["text"][:40000] for d in data["train"][:400] if len(d["text"].split()) > 2500][:8]
for s in longs:
    full = ENC_FN(NORM_FN(s)); h, t, nw = P4.encode_head_tail(s, NORM_FN, ENC_FN, 300, 900)
    assert t is not None and list(full[:len(h)]) == list(h) and list(full[-len(t):]) == list(t), "head/tail mismatch"
print(f"head/tail check on {len(longs)} long cases: prefix + suffix identical to full encoding")

# (d) tokenise + chunk everything (all CPU cores) -> cache used by every job
CHUNK_LEN, N_HEAD, N_TAIL = 512, 1, 7         # 1 head chunk + 7 tail chunks of 512 tokens (incl. <s>) = up to ~4k tokens/case
# RetNet pretraining saw every document as [<s>] + text (positions restart at 0 on <s>); every chunk gets the same <s>.
ADD_BOS = True
BOS_ID = 2 if tok.kind == "old" else 1
SPEC = dict(arch=ARCH, tok_py=TOK_PY, tok_path=TOK_PATH, norm=NORM, chunk_len=CHUNK_LEN, n_head=N_HEAD, n_tail=N_TAIL,
            bos=BOS_ID if ADD_BOS else None)
CACHE = WORK / "cache.pkl"
t0 = time.time()
if CACHE.exists() and pickle.load(open(CACHE, "rb"))["spec"] == SPEC:
    splits = pickle.load(open(CACHE, "rb"))["splits"]; print("re-using", CACHE)
else:
    splits = P4.prepare(data, SPEC, CACHE)
print(f"tokenised + chunked in {time.time() - t0:.0f}s")
display(pd.DataFrame(P4.chunk_stats(splits, SPEC)).T)
assert [d["id"] for d in splits["test"]] == [d["id"] for d in data["test"]]
del data

# (e) the encoder in CAUSAL mode must reproduce the Phase-1 forward pass exactly
J0 = dict(arch=ARCH, model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT)
base = P2.load_backbone(J0).eval()
print(f"Phase-1 model: {sum(p.numel() for p in base.parameters()) / 1e6:.2f}M params")
ids = torch.as_tensor(splits["dev"][0]["chunks"][0][:64].astype(np.int64))[None]
am = torch.ones_like(ids, dtype=torch.bool)
with torch.no_grad():
    tg = P2.RetTagger(base, 2, attn="causal", dropout=0.0).eval()
    S0, pd0 = base.init_state(1, ids.device)
    ref_h = base.hidden(ids, torch.zeros_like(ids), torch.arange(ids.size(1))[None], S0, pd0)[0]; mine = tg.hidden(ids, am)
diff = (ref_h - mine).abs().max().item(); print("encoder vs Phase-1 forward, max |diff| =", diff); assert diff < 1e-3
del base, tg


{"train_rows": 37387, "train_kept": 31677, "train": {"dropped_duplicate_text": 5710, "duplicates_with_conflicting_label": 0}, "train_labels": {"0": 18552, "1": 13125}, "dev_kept": 994, "dev": {"dropped_overlap_with_test_or_dev": 1517, "dropped_duplicate_text": 994, "duplicates_with_conflicting_label": 0}, "dev_labels": {"0": 497, "1": 497}, "test": 1517}
train 37387 rows -> 31677 unique cases {0: 18552, 1: 13125} | dev -> 994 clean cases {0: 497, 1: 497} | test 1517 cases (labels never used)
tokenizer kind: old | normalisation: clean | vocab 32000
checking against your original a1_tokenizer.encode
token-id check on 8 case texts: 8 identical
head/tail check on 8 long cases: prefix + suffix identical to full encoding
[prepare] train: 31677 cases tokenised in 52s
[prepare] dev: 994 cases tokenised in 4s
[prepare] test: 1517 cases tokenised in 5s
tokenised + chunked in 62s


,cases,words_p50,words_p90,chunks_mean,chunks_max,tokens_kept_p50,truncated_pct,labels
train,31677,2030,6431,5.15,8,2491,29.1,"{0: 18552, 1: 13125}"
dev,994,2821,6922,6.58,8,3819,45.5,"{0: 497, 1: 497}"
test,1517,2940,7302,6.64,8,3954,47.6,-


Phase-1 model: 137.90M params
encoder vs Phase-1 forward, max |diff| = 0.0


In [5]:
# ============================================================ 3. EXPERIMENTS (2 GPUs -> 2 queues)
COMMON = dict(
    arch=ARCH, cache=str(CACHE), model_py=MODEL_PY, phase1_ckpt=PHASE1_CKPT,
    attn="bidir",             # 'bidir' (every token sees both sides; symmetric decay) | 'causal' (= Phase-1 computation)
    doc_pool="bigru",         # 'bigru' (BiGRU over chunks + attention/max pooling) | 'attn' (pooling only)
    rnn_hidden=256, dropout=0.1,
    train_chunks=4,           # chunks per case while training (always the last one); prediction uses all stored chunks
    docs_per_step=4, eval_chunk_bs=64,
    lowrank_targets=["wq", "wk", "wv", "wg", "wo", "gate", "up", "down"],   # LoRMA / AdaLoRA / BitFit-bias targets
    adapter_targets=["wo", "down"],                  # Houlsby adapters after the retention output and the FFN output
    kd_student_layers=[0, 2, 4, 7, 9, 11],           # KD student: 6 of the 12 Phase-1 layers
    head_lr=1e-3, peft_lr=1e-3, lr=5e-5, weight_decay=0.01, warmup=0.05,
    epochs=2,                 # upper bound - the time budget usually ends training first (LR decay is fitted to it)
    time_budget_h=None,       # set per job below from GROUP
    calib_steps=40, n_evals=4, log_every=200,
    class_weight="balanced", seed=781,
    rank=16, alpha=32, lowrank_dropout=0.05, ada_init_r=12, ada_target_r=8, ada_orth_reg=0.1,
    adapter_dim=64, partial_top_layers=4, gradual_schedule=[[0.0, 0], [0.15, 2], [0.35, 4], [0.55, 6]],
    kd_temperature=2.0, kd_alpha=0.5,
    grad_ckpt=False, ckpt_micro=4,  # gradient checkpointing switches itself on after a CUDA OOM
    section_ablation=False, save_model=False, quick=QUICK_TEST, quick_docs=64,
)
#               name           method     overrides
JOBS = {
  "lorma":   ("lorma",   dict(peft_lr=3e-4, save_model=True, section_ablation=True)),   # compulsory + KD teacher (LR 1e-3 collapsed in P3)
  "kd":      ("kd",      dict(kd_teacher_dir=str(OUT / "lorma"))),                     # compulsory: 6-layer student
  "adalora": ("adalora", dict()),
  "bitfit":  ("bitfit",  dict(peft_lr=5e-4)),
  "adapter": ("adapter", dict()),
  "partial": ("partial", dict()),
  "gradual": ("gradual", dict()),
  "frozen":  ("frozen",  dict(hours_scale=0.5)),                                       # probe: Phase-1 features as-is
  "full":    ("full",    dict(lr=2e-5)),                                               # optional extra (not in a group)
}
GROUPS = {   # queues (one per GPU; kd after lorma on the SAME queue) , hours per job
  "ALL": ([["lorma", "kd", "adapter", "frozen"], ["adalora", "bitfit", "partial", "gradual"]], 2.2),
  "A":   ([["lorma", "kd"], ["adalora", "bitfit"]], 4.5),
  "B":   ([["adapter", "frozen"], ["partial", "gradual"]], 4.5),
}
QUEUES, JOB_HOURS = GROUPS[GROUP]
RUN_ONLY = None           # e.g. ["lorma", "kd"] to (re)run only those jobs
if RUN_ONLY:
    QUEUES = [[n for n in q if n in RUN_ONLY] for q in QUEUES]
print("group", GROUP, "| queues:", QUEUES, "| hours per job:", JOB_HOURS)

n_gpu = torch.cuda.device_count(); print("GPUs:", n_gpu)
if n_gpu < 2: QUEUES = [sum(QUEUES, [])]

def write_job(name, hours):
    meth, over = JOBS[name]
    over = dict(over); scale = over.pop("hours_scale", 1.0)
    J = dict(COMMON, name=name, method=meth, out_dir=str(OUT / name), **over)
    J["time_budget_h"] = hours * scale
    if QUICK_TEST: J.update(epochs=1, n_evals=2, calib_steps=10 ** 9, log_every=5, time_budget_h=0.5)
    Path(J["out_dir"]).mkdir(parents=True, exist_ok=True)
    p = Path(J["out_dir"]) / "job.json"; json.dump(J, open(p, "w"), indent=1)
    return p

def run_queue(names, gpu):
    for n in names:
        if (OUT / n / "results.json").exists():
            print(f"[skip] {n} already done"); continue
        left = SESSION_H - (time.time() - T_START) / 3600 - 0.5          # keep 30 min for the summary + saving the output
        scale = JOBS[n][1].get("hours_scale", 1.0)
        hours = min(JOB_HOURS, (left - 0.25) / scale)                   # ~15 min per job for loading + test prediction
        if not QUICK_TEST and hours < 0.5:
            print(f"[gpu{gpu}] !!! not enough session time left for {n} ({left:.2f} h) - skipped; rerun it with RUN_ONLY"); continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu) if n_gpu else "", PYTHONUNBUFFERED="1")
        p = subprocess.Popen([sys.executable, str(WORK / "p4_cjpe.py"), str(write_job(n, hours))], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            if "Warning" not in line and "warn(" not in line: print(f"[gpu{gpu}] {line}", end="")
        p.wait()
        if p.returncode != 0:
            print(f"[gpu{gpu}] !!! job {n} FAILED (see {OUT / n / 'log.txt'} and the lines above)")

T0 = time.time()
threads = [threading.Thread(target=run_queue, args=(q, g)) for g, q in enumerate(QUEUES)]
for t in threads: t.start()
for t in threads: t.join()
print(f"all jobs finished in {(time.time() - T0) / 60:.1f} min")


group B | queues: [['adapter', 'frozen'], ['partial', 'gradual']] | hours per job: 4.5
GPUs: 2
[gpu0] [20:22:35][adapter] arch=ret method=adapter doc_pool=bigru attn=bidir device=cuda | train 31677 dev 994 test 1517 cases | budget 4.50 h
[gpu1] [20:22:35][partial] arch=ret method=partial doc_pool=bigru attn=bidir device=cuda | train 31677 dev 994 test 1517 cases | budget 4.50 h
[gpu0] [20:22:38][adapter] params total=141.93M (encoder 140.28M) trainable=4.037M | PEFT tensors 96
[gpu1] [20:22:38][partial] params total=139.56M (encoder 137.90M) trainable=39.432M | PEFT tensors 0
[gpu1] [20:22:55][partial] calibration: 0.341 s/step (4 cases x <= 4 chunks) | ~0.9 min per dev eval | steps 15840 planned -> 15840 fit the 4.50 h budget (= 2.00 epochs of 31677 cases)
[gpu0] [20:23:00][adapter] calibration: 0.489 s/step (4 cases x <= 4 chunks) | ~1.3 min per dev eval | steps 15840 planned -> 15840 fit the 4.50 h budget (= 2.00 epochs of 31677 cases)
[gpu1] [20:23:54][partial] {"step": 200, "total

In [6]:
# ============================================================ 4. SUMMARY + SUBMISSIONS (+ an ensemble)
import zipfile
SUB = Path("/kaggle/working/submissions" + ("_quick" if QUICK_TEST else "")); SUB.mkdir(exist_ok=True)
test_docs = splits["test"]
if QUICK_TEST: test_docs = P4.subsample(test_docs, max(20, COMMON["quick_docs"] // 2), 3)
rows, sec = [], None
for n in JOBS:
    rp = OUT / n / "results.json"
    if not rp.exists(): continue
    r = json.load(open(rp)); tr = r["train"]
    z = OUT / n / f"submission_{n}.zip"
    with zipfile.ZipFile(z) as zz:
        assert zz.namelist() == ["prediction.json"]
        P4.check_prediction(json.loads(zz.read("prediction.json")), test_docs)
    shutil.copy(z, SUB / z.name)
    rows.append(dict(exp=n, method=r["method"], dev_macroF1=round(r["dev"]["macro_f1"], 4), thr=r["dev"]["thr"],
                     dev_F1_at_05=round(r["dev_at_0.5"]["macro_f1"], 4), dev_AUC=round(r["dev_auc"], 4),
                     dev_acc=round(r["dev"]["accuracy"], 4), chunks=r["infer_section"], epochs_seen=round(tr["epochs"], 2), best_step=r["best_step"],
                     train_h=round(tr["train_secs"] / 3600, 2), params_M=round(r["params_total"] / 1e6, 2),
                     trainable_M=round(r["params_trainable"] / 1e6, 3), peak_GB=round(tr["peak_mem_gb"] or 0, 2),
                     ms_per_case=round(r["test_infer_ms_per_case"], 1), test_0_1=str(r["test_label_counts"]),
                     UPLOAD="NO - collapsed" if r["collapsed"] else "yes"))
    if "sections" in r: sec = r["sections"]
df = pd.DataFrame(rows).sort_values("dev_macroF1", ascending=False) if rows else pd.DataFrame()
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(df)
df.to_csv(WORK / ("summary_quick.csv" if QUICK_TEST else "summary.csv"), index=False)
if sec:
    print("SECTION ABLATION (LoRMA model, dev): which part of the judgment carries the outcome signal?")
    display(pd.DataFrame(sec).T)

# extra submission: average P(1) of the best 3 runs (an ensemble - NOT a new technique), threshold tuned on dev
ENS = [e for e in df["exp"].tolist() if e != "frozen" and (OUT / e / "test_probs.npz").exists()][:3] if len(df) else []
if len(ENS) >= 2:
    dv = [np.load(OUT / e / "dev_probs.npz") for e in ENS]; ts = [np.load(OUT / e / "test_probs.npz") for e in ENS]
    assert all((d["ids"] == dv[0]["ids"]).all() for d in dv) and all((t["ids"] == ts[0]["ids"]).all() for t in ts)
    pdv = np.mean([d["probs"] for d in dv], 0); pts = np.mean([t["probs"] for t in ts], 0)
    thr, f1 = P4.tune_threshold(dv[0]["y"], pdv)
    pred = [dict(id=str(i), pred=int(p >= thr)) for i, p in zip(ts[0]["ids"], pts)]
    P4.check_prediction(pred, test_docs)
    P4.write_zip(pred, SUB / ("submission_ensemble_" + "_".join(ENS) + ".zip"))
    print(f"ensemble of {ENS}: dev macro-F1 {f1:.4f} @ {thr:.2f} | test 0/1 = {dict(pd.Series([p['pred'] for p in pred]).value_counts())}")
print("submission zips:", sorted(p.name for p in SUB.glob("*.zip")), "->", SUB)
if QUICK_TEST: print("\nQUICK TEST ONLY - do NOT submit these (they cover only a few test cases). Set QUICK_TEST=False and Save & Run All (Commit).")


,exp,method,dev_macroF1,thr,dev_F1_at_05,dev_AUC,dev_acc,chunks,epochs_seen,best_step,train_h,params_M,trainable_M,peak_GB,ms_per_case,test_0_1,UPLOAD
1,partial,partial,0.7364,0.11,0.6389,0.8091,0.7364,all,2.0,15840,1.92,139.56,39.432,3.52,124.4,"{'0': 715, '1': 802}",yes
2,gradual,gradual,0.7261,0.33,0.6718,0.8024,0.7274,all,2.0,15840,1.91,139.56,58.319,4.96,123.4,"{'0': 858, '1': 659}",yes
3,frozen,frozen,0.6017,0.40,0.5557,0.6382,0.6026,last4,2.0,15840,1.19,139.56,1.658,2.10,68.2,"{'0': 720, '1': 797}",yes
0,adapter,adapter,0.3333,0.50,0.3333,0.4698,0.5000,all,2.0,3960,2.49,141.93,4.037,6.51,119.5,{'0': 1517},NO - collapsed


ensemble of ['partial', 'gradual', 'adapter']: dev macro-F1 0.7392 @ 0.30 | test 0/1 = {0: np.int64(770), 1: np.int64(747)}
submission zips: ['submission_adapter.zip', 'submission_ensemble_partial_gradual_adapter.zip', 'submission_frozen.zip', 'submission_gradual.zip', 'submission_partial.zip'] -> /kaggle/working/submissions
